# code-server on Colab — plug and play

**Runtime → Run all.** No clone, no Drive, no patches to code-server. The only
optional input is an ngrok token; without one it falls back to a Cloudflare quick
tunnel that needs no account.

Nothing here renames processes or spoofs traffic — that layer is not what Colab
enforces on. See `README.md` (written to disk by cell 4) for the layer-by-layer
breakdown. What this does fix: rotating URLs, blank pages, and VMs left idle.

Colab's FAQ does not list remote-IDE use as supported. It works; the runtime is
ephemeral and account-level throttling is always possible.

## Flow, and where the logs are

```
install.sh                     (cell 5, slow: 1–3 min first run)
  apt-get runtime libs  →  code-server tarball  →  ngrok  →  cloudflared

supervise.sh                   (cell 6, returns immediately)
  └─ serve.sh                  (cell 7 watches this live)
       clear stale pidfiles
       port check
       code-server start  →  poll /healthz           (5–90 s)
       tunnel start       →  poll for public URL     (10–60 s)
       write credentials.txt  →  print URL + password
  └─ then: restart on crash, stop when idle
```

Every stage appends to one log:

| File | Contents |
|---|---|
| `/content/.code-server/supervise.out` | **the progress log** — one line per stage |
| `/content/.code-server/code-server.log` | editor server; startup failures land here |
| `/content/.code-server/cloudflared.log` | tunnel registration and the public URL |
| `/content/.code-server/ngrok.log` | JSON; errors like `ERR_NGROK_4018` |
| `/content/.code-server/credentials.txt` | final `url=` / `password=` |

If cell 7 seems stuck, read `supervise.out`: the last line names the stage.

## 1. Environment

In [ ]:
import os, sys, platform, pathlib, shutil

IN_COLAB = "google.colab" in sys.modules or pathlib.Path("/content").is_dir()
BASE = pathlib.Path("/content/colab-cs")

print("colab      :", IN_COLAB)
print("platform   :", platform.platform())
print("machine    :", platform.machine())
print("python     :", platform.python_version())
try:
    print("free disk  :", shutil.disk_usage("/content").free // 2**30, "GiB")
except OSError:
    pass
print("install to :", BASE)

## 2. Configuration

Every value is optional. `TUNNEL` stays `auto` unless you force one:

* `auto` — ngrok if it has credentials, else cloudflared (no account needed)
* `ngrok` — needs `NGROK_TOKEN` (or a Colab Secret of that name); fastest
* `cloudflared` — no account, slightly slower to register, URL changes per run
* `none` — loopback only, for debugging

In [ ]:
import os

# ── edit here if you want ─────────────────────────────────────────────────────
NGROK_TOKEN    = ""        # or add a Colab Secret named NGROK_TOKEN
TUNNEL         = ""        # "" = auto | "ngrok" | "cloudflared" | "none"
NGROK_DOMAIN   = ""        # reserved domain => stable URL, e.g. "me.ngrok-free.app"
NGROK_AUTH     = ""        # optional edge gate "user:pass" (rejects scanners)
IDLE_TIMEOUT   = 1800      # seconds with zero connections before shutdown
MAX_RUNTIME    = 39600     # seconds before forced shutdown (11 h)
STATE_ON_DRIVE = False     # keep extensions and settings between runtimes
CS_EXTENSIONS  = ""        # comma-separated IDs, installed once
# ─────────────────────────────────────────────────────────────────────────────

token = NGROK_TOKEN.strip()
token_source = "config cell" if token else ""
if not token:
    try:
        from google.colab import userdata
        token = (userdata.get("NGROK_TOKEN") or "").strip()
        token_source = "Colab secret" if token else ""
    except Exception:
        pass

os.environ["NGROK_TOKEN"] = token
os.environ["CS_TUNNEL"] = TUNNEL
os.environ["CS_IDLE_TIMEOUT"] = str(IDLE_TIMEOUT)
os.environ["CS_MAX_RUNTIME"] = str(MAX_RUNTIME)
if NGROK_DOMAIN: os.environ["NGROK_DOMAIN"] = NGROK_DOMAIN
if NGROK_AUTH:   os.environ["NGROK_BASIC_AUTH"] = NGROK_AUTH
if CS_EXTENSIONS: os.environ["CS_EXTENSIONS"] = CS_EXTENSIONS

if STATE_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["CS_STATE_DIR"] = "/content/drive/MyDrive/.code-server"
    os.environ["CS_CACHE_DIR"] = "/content/drive/MyDrive/.code-server/cache"

print("tunnel      :", TUNNEL or "auto")
print("ngrok token :", f"yes ({token_source})" if token else "no -> cloudflared fallback")
print("state dir   :", os.environ.get("CS_STATE_DIR", "/content/.code-server"))

## 3. Write the toolkit

In [ ]:
import base64, pathlib, stat

FILES = {
    'lib/common.sh': (
        'IyEvdXNyL2Jpbi9lbnYgYmFzaAojIGNvbGFiL2xpYi9jb21tb24uc2gg4oCUIHNoYXJlZCBoZWxwZXJzIGZvciB0aGUgQ29sYWIgc2NyaXB0cy4KIwojIFNvdXJjZSB0aGlzIGZpbGUsIGRvIG5vdCBleGVjdXRlIGl0OgojICAgc291cmNlICIkKGRpcm5hbWUgIiQwIikvbGliL2NvbW1vbi5zaCIKIwojIEV2ZXJ5dGhpbmcgaGVyZSBpcyBkZWxpYmVyYXRlbHkgc2lkZS1lZmZlY3QgZnJlZSBhcGFydCBmcm9tIGluc3RhbGxpbmcgYW4gRVJSCiMgdHJhcCBhbmQgdGhlIGNvbG91ciBjb25zdGFudHMuCgppZiBbWyAtbiAiJHtDU19DT01NT05fTE9BREVEOi19IiBdXTsgdGhlbgogIHJldHVybiAwCmZpCkNTX0NPTU1PTl9MT0FERUQ9MQoKc2V0IC1FZXVvIHBpcGVmYWlsCgojIHNoZWxsY2hlY2sgZGlzYWJsZT1TQzIxMjggICMgQkFTSF9TT1VSQ0UgaXMgdGhlIGZpbGUgYmVpbmcgc291cmNlZApyZWFkb25seSBDU19MSUJfRElSPSIkKGNkIC0tICIkKGRpcm5hbWUgLS0gIiR7QkFTSF9TT1VSQ0VbMF19IikiICYmIHB3ZCAtUCkiCnJlYWRvbmx5IENTX1JFUE9fUk9PVD0iJChjZCAtLSAiJHtDU19MSUJfRElSfS8uLi8uLiIgJiYgcHdkIC1QKSIKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIG91dHB1dCAtLS0tCmlmIFtbIC10IDEgXV07IHRoZW4KICBDU19DX1JFU0VUPSQnXDAzM1swbScKICBDU19DX0JMVUU9JCdcMDMzWzE7MzRtJwogIENTX0NfR1JFRU49JCdcMDMzWzE7MzJtJwogIENTX0NfUkVEPSQnXDAzM1sxOzMxbScKICBDU19DX1lFTExPVz0kJ1wwMzNbMTszM20nCiAgQ1NfQ19ESU09JCdcMDMzWzJtJwplbHNlCiAgQ1NfQ19SRVNFVD0nJyBDU19DX0JMVUU9JycgQ1NfQ19HUkVFTj0nJyBDU19DX1JFRD0nJyBDU19DX1lFTExPVz0nJyBDU19DX0RJTT0nJwpmaQoKbG9nKCkgIHsgcHJpbnRmICclcz09PiVzICVzXG4nICAgIiRDU19DX0JMVUUiICAgIiRDU19DX1JFU0VUIiAiJCoiOyB9Cm9rKCkgICB7IHByaW50ZiAnJXMgb2slcyAlc1xuJyAgICIkQ1NfQ19HUkVFTiIgICIkQ1NfQ19SRVNFVCIgIiQqIjsgfQp3YXJuKCkgeyBwcmludGYgJyVzd2FybiVzICVzXG4nICAiJENTX0NfWUVMTE9XIiAiJENTX0NfUkVTRVQiICIkKiIgPiYyOyB9CmRpZSgpICB7IHByaW50ZiAnJXNlcnIlcyAlc1xuJyAgICIkQ1NfQ19SRUQiICAgICIkQ1NfQ19SRVNFVCIgIiQqIiA+JjI7IGV4aXQgMTsgfQpkaW0oKSAgeyBwcmludGYgJyVzJXMlc1xuJyAgICAgICAiJENTX0NfRElNIiAgICAiJCoiICIkQ1NfQ19SRVNFVCI7IH0KCiMgUmVwb3J0IHRoZSBmYWlsaW5nIGxpbmUgaW5zdGVhZCBvZiBkeWluZyBzaWxlbnRseS4gRXhlbXB0IGZyb20gdGhlIHVzdWFsCiMgYCYmYC9gfHxgL2BpZmAgY2FzZXMsIHNvIGl0IG9ubHkgZmlyZXMgb24gZ2VudWluZWx5IHVuZXhwZWN0ZWQgZmFpbHVyZXMuCmNzX29uX2VycigpIHsKICBsb2NhbCBlYz0kPyBsaW5lPSQxCiAgcHJpbnRmICclc2VyciVzICVzOiVzIGV4aXRlZCB3aXRoICVkXG4nIFwKICAgICIkQ1NfQ19SRUQiICIkQ1NfQ19SRVNFVCIgIiQoYmFzZW5hbWUgLS0gIiR7QkFTSF9TT1VSQ0VbMl06LT99IikiICIkbGluZSIgIiRlYyIgPiYyCiAgZXhpdCAiJGVjIgp9CnRyYXAgJ2NzX29uX2VyciAiJExJTkVOTyInIEVSUgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gc3lzdGVtIC0tLS0KY3NfaXNfcm9vdCgpIHsgW1sgIiQoaWQgLXUpIiAtZXEgMCBdXTsgfQoKY3Nfcm9vdCgpIHsKICBpZiBjc19pc19yb290OyB0aGVuICIkQCI7IGVsc2Ugc3VkbyAtLSAiJEAiOyBmaQp9Cgpjc19uZWVkKCkgewogIGxvY2FsIGNtZD0kMSBoaW50PSR7MjotfQogIGNvbW1hbmQgLXYgIiRjbWQiID4vZGV2L251bGwgMj4mMSB8fCBkaWUgIm1pc3NpbmcgY29tbWFuZDogJHtjbWR9JHtoaW50OisgKCR7aGludH0pfSIKfQoKY3Nfb3NfZmllbGQoKSB7CiAgKCAjIHNoZWxsY2hlY2sgZGlzYWJsZT1TQzEwOTEKICAgIC4gL2V0Yy9vcy1yZWxlYXNlIDI+L2Rldi9udWxsIHx8IGV4aXQgMAogICAgcHJpbnRmICclcycgIiR7ITE6LX0iCiAgKQp9CgojIFVidW50dSAyNC4wNCsgcmVuYW1lZCBsaWJhc291bmQyIC0+IGxpYmFzb3VuZDJ0NjQuIENvbGFiIHN0aWxsIHNoaXBzIDIyLjA0LAojIGJ1dCB0aGUgaW1hZ2UgY2hhbmdlcyB3aXRob3V0IG5vdGljZSwgc28gZGV0ZWN0IGluc3RlYWQgb2YgaGFyZGNvZGluZy4KY3NfYXNvdW5kX3BrZygpIHsKICBjYXNlICIkKGNzX29zX2ZpZWxkIFZFUlNJT05fSUQpIiBpbgogICAgMjQuMDQqfDI1Lip8MjYuKnwyWzctOV0uKikgcHJpbnRmICdsaWJhc291bmQydDY0JyA7OwogICAgKikgICAgICAgICAgICAgICAgICAgICAgICAgcHJpbnRmICdsaWJhc291bmQyJyA7OwogIGVzYWMKfQoKY3NfYXJjaCgpIHsKICBjYXNlICIkKHVuYW1lIC1tKSIgaW4KICAgIHg4Nl82NHxhbWQ2NCkgIHByaW50ZiAnYW1kNjQnIDs7CiAgICBhYXJjaDY0fGFybTY0KSBwcmludGYgJ2FybTY0JyA7OwogICAgKikgZGllICJ1bnN1cHBvcnRlZCBhcmNoaXRlY3R1cmU6ICQodW5hbWUgLW0pIiA7OwogIGVzYWMKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gc3RhdGUgZGlyIC0tLS0KY3Nfc3RhdGVfZGlyKCkgewogIGxvY2FsIGQ9IiR7Q1NfU1RBVEVfRElSOi0vY29udGVudC8uY29kZS1zZXJ2ZXJ9IgogIG1rZGlyIC1wIC0tICIkZCIgfHwgZGllICJjYW5ub3QgY3JlYXRlIHN0YXRlIGRpcjogJHtkfSIKICBjaG1vZCA3MDAgLS0gIiRkIiAyPi9kZXYvbnVsbCB8fCB0cnVlCiAgcHJpbnRmICclcycgIiRkIgp9Cgpjc19jYWNoZV9kaXIoKSB7CiAgbG9jYWwgZD0iJHtDU19DQUNIRV9ESVI6LSQoY3Nfc3RhdGVfZGlyKS9jYWNoZX0iCiAgbWtkaXIgLXAgLS0gIiRkIiB8fCBkaWUgImNhbm5vdCBjcmVhdGUgY2FjaGUgZGlyOiAke2R9IgogIHByaW50ZiAnJXMnICIkZCIKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gbG9jayAtLS0tCkNTX0xPQ0tfUEFUSD0nJwoKY3NfbG9jaygpIHsKICBsb2NhbCBuYW1lPSR7MTotbG9ja30gc2QgZgogIHNkPSIkKGNzX3N0YXRlX2RpcikiCiAgZj0iJHtzZH0vJHtuYW1lfS5sb2NrIgogIGlmIGNvbW1hbmQgLXYgZmxvY2sgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICBleGVjIDk+IiRmIiB8fCBkaWUgImNhbm5vdCBvcGVuIGxvY2s6ICR7Zn0iCiAgICBmbG9jayAtbiA5IHx8IGRpZSAiYW5vdGhlciBpbnN0YW5jZSBpcyBhbHJlYWR5IHJ1bm5pbmcgKCR7Zn0pIgogICAgQ1NfTE9DS19QQVRIPSIkZiIKICBlbHNlCiAgICBsb2NhbCBkPSIke2Z9LmQiCiAgICBta2RpciAtLSAiJGQiIDI+L2Rldi9udWxsIHx8IGRpZSAiYW5vdGhlciBpbnN0YW5jZSBpcyBhbHJlYWR5IHJ1bm5pbmcgKCR7ZH0pIgogICAgcHJpbnRmICclc1xuJyAiJCQiID4iJHtkfS9waWQiCiAgICBDU19MT0NLX1BBVEg9IiRkIgogIGZpCn0KCmNzX3VubG9jaygpIHsKICBbWyAtbiAiJHtDU19MT0NLX1BBVEg6LX0iIF1dIHx8IHJldHVybiAwCiAgaWYgW1sgLWQgIiRDU19MT0NLX1BBVEgiIF1dOyB0aGVuCiAgICBybSAtcmYgLS0gIiRDU19MT0NLX1BBVEgiCiAgZWxzZQogICAgOiA+IiRDU19MT0NLX1BBVEgiIDI+L2Rldi9udWxsIHx8IHRydWUKICBmaQogIENTX0xPQ0tfUEFUSD0nJwogIHJldHVybiAwCn0KCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gcGlkZmlsZXMgLS0tLQpjc19yZWFkX3BpZCgpIHsgW1sgLXMgIiR7MTotfSIgXV0gJiYgY2F0IC0tICIkMSIgMj4vZGV2L251bGwgfHwgcHJpbnRmICcnOyB9CmNzX3dyaXRlX3BpZCgpIHsgcHJpbnRmICclc1xuJyAiJDIiID4iJDEiOyB9Cgpjc19hbGl2ZSgpIHsKICBsb2NhbCBwCiAgcD0iJChjc19yZWFkX3BpZCAiJHsxOi19IikiCiAgW1sgLW4gIiRwIiBdXSB8fCByZXR1cm4gMQogIGtpbGwgLTAgIiRwIiAyPi9kZXYvbnVsbAp9Cgpjc19zdG9wX3BpZGZpbGUoKSB7CiAgbG9jYWwgZj0kMSBsYWJlbD0kezI6LXByb2Nlc3N9IHAgaQogIHA9IiQoY3NfcmVhZF9waWQgIiRmIikiCiAgcm0gLWYgLS0gIiRmIgogIFtbIC1uICIkcCIgXV0gfHwgcmV0dXJuIDAKICBraWxsIC0wICIkcCIgMj4vZGV2L251bGwgfHwgcmV0dXJuIDAKICBsb2cgInN0b3BwaW5nICR7bGFiZWx9IChwaWQgJHtwfSkiCiAga2lsbCAtVEVSTSAiJHAiIDI+L2Rldi9udWxsIHx8IHRydWUKICBmb3IgaSBpbiB7MS4uMTV9OyBkbwogICAga2lsbCAtMCAiJHAiIDI+L2Rldi9udWxsIHx8IHJldHVybiAwCiAgICBzbGVlcCAwLjMKICBkb25lCiAga2lsbCAtS0lMTCAiJHAiIDI+L2Rldi9udWxsIHx8IHRydWUKICByZXR1cm4gMAp9CgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBodHRwIC0tLS0KY3Nfd2FpdF9odHRwKCkgewogIGxvY2FsIHVybD0kMSB0aW1lb3V0PSR7MjotNjB9IGkKICBmb3IgKChpID0gMDsgaSA8IHRpbWVvdXQ7IGkrKykpOyBkbwogICAgIyAtLW5vcHJveHk6IENvbGFiL0p1cHl0ZXIgc29tZXRpbWVzIGV4cG9ydHMgaHR0cF9wcm94eSwgd2hpY2ggd291bGQgc2VuZAogICAgIyBsb29wYmFjayBjaGVja3MgdGhyb3VnaCBhIHByb3h5IHRoYXQgY2Fubm90IHJlYWNoIDEyNy4wLjAuMS4KICAgIGlmIGN1cmwgLWZzUyAtLW1heC10aW1lIDIgLS1ub3Byb3h5ICcqJyAiJHVybCIgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICAgIHJldHVybiAwCiAgICBmaQogICAgIyBGaXJzdCBzdGFydCBvbiBhIGZyZXNoIFZNIGlzIG5vdCBpbnN0YW50OyBlbWl0IGEgaGVhcnRiZWF0IHNvIGEgc2xvdyBydW4KICAgICMgaXMgZGlzdGluZ3Vpc2hhYmxlIGZyb20gYSBodW5nIG9uZSB3aGVuIHJlYWRpbmcgdGhlIGxvZyBhZnRlcndhcmRzLgogICAgaWYgKChpID4gMCAmJiBpICUgMTUgPT0gMCkpOyB0aGVuCiAgICAgIGRpbSAiICB3YWl0aW5nIGZvciAke3VybH0gKCR7aX1zKSIKICAgIGZpCiAgICBzbGVlcCAxCiAgZG9uZQogIHJldHVybiAxCn0KCmNzX3BvcnRfaW5fdXNlKCkgewogIGxvY2FsIHA9JDEKICBpZiBjb21tYW5kIC12IHNzID4vZGV2L251bGwgMj4mMTsgdGhlbgogICAgc3MgLWx0bkggMj4vZGV2L251bGwgfCBhd2sgLXYgcGF0PSI6JHtwfVwkIiAnJDQgfiBwYXQgeyBmb3VuZCA9IDEgfSBFTkQgeyBleGl0ICFmb3VuZCB9JwogIGVsc2UKICAgIChleGVjIDM8PiIvZGV2L3RjcC8xMjcuMC4wLjEvJHtwfSIpIDI+L2Rldi9udWxsCiAgZmkKfQoKIyBFc3RhYmxpc2hlZCBjb25uZWN0aW9ucyBvbiBhIGxvY2FsIHBvcnQuIFVzZWQgYnkgdGhlIGlkbGUgd2F0Y2hkb2c6IGNvdW50aW5nCiMgc29ja2V0cyBpcyBjaGVhcCBhbmQgbmVlZHMgbm90aGluZyBmcm9tIHRoZSBlZGl0b3IgaXRzZWxmLgpjc19hY3RpdmVfY29ubnMoKSB7CiAgbG9jYWwgcD0kMSBuCiAgaWYgY29tbWFuZCAtdiBzcyA+L2Rldi9udWxsIDI+JjE7IHRoZW4KICAgIG49IiQoc3MgLUh0biBzdGF0ZSBlc3RhYmxpc2hlZCAic3BvcnQgPSA6JHtwfSIgMj4vZGV2L251bGwgfCB3YyAtbCB8IHRyIC1kICcgJykiCiAgICBwcmludGYgJyVzJyAiJHtuOi0wfSIKICAgIHJldHVybiAwCiAgZmkKICBuPSIkKHB5dGhvbjMgLWMgJwppbXBvcnQgc3lzCnBvcnQgPSBpbnQoc3lzLmFyZ3ZbMV0pCndhbnQgPSAiJTA0WCIgJSBwb3J0Cm4gPSAwCmZvciBmIGluICgiL3Byb2MvbmV0L3RjcCIsICIvcHJvYy9uZXQvdGNwNiIpOgogICAgdHJ5OgogICAgICAgIHJvd3MgPSBvcGVuKGYpLnJlYWQoKS5zcGxpdGxpbmVzKClbMTpdCiAgICBleGNlcHQgT1NFcnJvcjoKICAgICAgICBjb250aW51ZQogICAgZm9yIHIgaW4gcm93czoKICAgICAgICBjID0gci5zcGxpdCgpCiAgICAgICAgaWYgbGVuKGMpID4gMyBhbmQgY1sxXS5zcGxpdCgiOiIpWzFdLnVwcGVyKCkgPT0gd2FudCBhbmQgY1szXSA9PSAiMDEiOgogICAgICAgICAgICBuICs9IDEKcHJpbnQobikKJyAiJHAiIDI+L2Rldi9udWxsIHx8IHRydWUpIgogIHByaW50ZiAnJXMnICIke246LTB9Igp9CgojIEEgY29kZS1zZXJ2ZXIgcHJvY2VzcyBkb2VzIG5vdCBrZWVwICJjb2RlLXNlcnZlciAtLWJpbmQtYWRkciIgaW4gaXRzIGFyZ3Y6IHRoZQojIGJpbi9jb2RlLXNlcnZlciB3cmFwcGVyIHJlLWV4ZWNzLCBzbyBhcmd2IGxvb2tzIGxpa2UKIyAgIC4uLi9saWIvY29kZS1zZXJ2ZXItPHZlcj4vbGliL25vZGUgLi4uL291dC9ub2RlL2VudHJ5LmpzIC0tYmluZC1hZGRyIDEyNy4wLjAuMTo4MDgwCiMgd2hpY2ggaXMgd2h5IHBhdHRlcm4tbWF0Y2hpbmcgcGtpbGwgbWlzc2VzIG9ycGhhbmVkIGluc3RhbmNlcy4gUmVzb2x2ZSB0aGUKIyBzb2NrZXQgb3duZXIgaW5zdGVhZC4KY3NfY21kbGluZSgpIHsKICB0ciAnXDAnICcgJyA8Ii9wcm9jLyR7MX0vY21kbGluZSIgMj4vZGV2L251bGwgfCBzZWQgJ3MvICokLy8nIHx8IHRydWUKfQoKY3NfcG9ydF9vd25lcl9waWRzKCkgewogIGxvY2FsIHA9JDEKICBpZiBjb21tYW5kIC12IHNzID4vZGV2L251bGwgMj4mMTsgdGhlbgogICAgc3MgLWx0bnBIIDI+L2Rldi9udWxsIHwgYXdrIC12IHBhdD0iOiR7cH1cJCIgJyQ0IH4gcGF0JyBcCiAgICAgIHwgZ3JlcCAtb0UgJ3BpZD1bMC05XSsnIHwgY3V0IC1kPSAtZjIgfCBzb3J0IC11IHx8IHRydWUKICAgIHJldHVybiAwCiAgZmkKICBpZiBjb21tYW5kIC12IGxzb2YgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICBsc29mIC10aVRDUDoiJHAiIC1zVENQOkxJU1RFTiAyPi9kZXYvbnVsbCB8IHNvcnQgLXUgfHwgdHJ1ZQogICAgcmV0dXJuIDAKICBmaQogIGlmIGNvbW1hbmQgLXYgZnVzZXIgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICBmdXNlciAtbiB0Y3AgIiRwIiAyPi9kZXYvbnVsbCB8IHRyIC1zICcgJyAnXG4nIHwgZ3JlcCAtRSAnXlswLTldKyQnIHwgc29ydCAtdSB8fCB0cnVlCiAgZmkKICByZXR1cm4gMAp9Cgpjc19wb3J0X3JlcG9ydCgpIHsKICBsb2NhbCBwPSQxIG93bmVycyBwaWQKICBvd25lcnM9IiQoY3NfcG9ydF9vd25lcl9waWRzICIkcCIpIgogIGlmIFtbIC16ICIkb3duZXJzIiBdXTsgdGhlbgogICAgd2FybiAicG9ydCAke3B9IGlzIGluIHVzZSBidXQgdGhlIG93bmVyIGNvdWxkIG5vdCBiZSBpZGVudGlmaWVkIgogICAgcmV0dXJuIDAKICBmaQogIGZvciBwaWQgaW4gJG93bmVyczsgZG8KICAgIHdhcm4gInBvcnQgJHtwfSBvd25lZCBieSBwaWQgJHtwaWR9OiAkKGNzX2NtZGxpbmUgIiRwaWQiIHwgY3V0IC1jMS0xMDApIgogIGRvbmUKfQoKIyBSZWNsYWltIHRoZSBwb3J0IGZyb20gYSBwcmV2aW91cyBydW4gb2YgdGhpcyBzdGFjay4gUmVmdXNlcyB0byB0b3VjaCBhbnl0aGluZwojIHRoYXQgZG9lcyBub3QgbG9vayBsaWtlIG91ciBvd24gcHJvY2Vzc2VzLgpjc19mcmVlX3BvcnQoKSB7CiAgbG9jYWwgcD0kMSBvd25lcnMgcGlkIGNtZCBpCiAgb3duZXJzPSIkKGNzX3BvcnRfb3duZXJfcGlkcyAiJHAiKSIKICBbWyAtbiAiJG93bmVycyIgXV0gfHwgcmV0dXJuIDAKCiAgZm9yIHBpZCBpbiAkb3duZXJzOyBkbwogICAgY21kPSIkKGNzX2NtZGxpbmUgIiRwaWQiKSIKICAgIGNhc2UgIiRjbWQiIGluCiAgICAgICplbnRyeS5qcyp8KmNvZGUtc2VydmVyKnwqbmdyb2sqfCpjbG91ZGZsYXJlZCopCiAgICAgICAgd2FybiAicmVjbGFpbWluZyBwb3J0ICR7cH0gZnJvbSBwaWQgJHtwaWR9OiAke2NtZDowOjgwfSIKICAgICAgICBraWxsIC1URVJNICIkcGlkIiAyPi9kZXYvbnVsbCB8fCB0cnVlCiAgICAgICAgOzsKICAgICAgKikKICAgICAgICB3YXJuICJwb3J0ICR7cH0gaXMgaGVsZCBieSBhbiB1bnJlbGF0ZWQgcHJvY2VzcyAocGlkICR7cGlkfSk6ICR7Y21kOjA6ODB9IgogICAgICAgIHJldHVybiAxCiAgICAgICAgOzsKICAgIGVzYWMKICBkb25lCgogIGZvciBpIGluIHsxLi4yMH07IGRvCiAgICBjc19wb3J0X2luX3VzZSAiJHAiIHx8IHJldHVybiAwCiAgICBzbGVlcCAwLjMKICBkb25lCiAgZm9yIHBpZCBpbiAkb3duZXJzOyBkbyBraWxsIC1LSUxMICIkcGlkIiAyPi9kZXYvbnVsbCB8fCB0cnVlOyBkb25lCiAgZm9yIGkgaW4gezEuLjEwfTsgZG8KICAgIGNzX3BvcnRfaW5fdXNlICIkcCIgfHwgcmV0dXJuIDAKICAgIHNsZWVwIDAuMwogIGRvbmUKICByZXR1cm4gMQp9CgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHBhc3N3b3JkIC0tLS0KIyBQZXJzaXN0ZWQgaW4gdGhlIHN0YXRlIGRpciBzbyBhIHJlc3RhcnQgZG9lcyBub3QgaW52YWxpZGF0ZSBhbiBvcGVuIHRhYi4KIyBDU19QQVNTV09SRCBvdmVycmlkZXMgd2l0aG91dCB0b3VjaGluZyBkaXNrLgpjc19wYXNzd29yZCgpIHsKICBsb2NhbCBzZD0kMSBmIHAKICBmPSIke3NkfS9wYXNzd29yZCIKICBpZiBbWyAtbiAiJHtDU19QQVNTV09SRDotfSIgXV07IHRoZW4KICAgIHByaW50ZiAnJXMnICIkQ1NfUEFTU1dPUkQiCiAgICByZXR1cm4gMAogIGZpCiAgaWYgW1sgLXMgIiRmIiBdXTsgdGhlbgogICAgY2F0IC0tICIkZiIKICAgIHJldHVybiAwCiAgZmkKICBwPSIkKExDX0FMTD1DIHRyIC1kYyAnQS1aYS16MC05JyA8L2Rldi91cmFuZG9tIDI+L2Rldi9udWxsIHwgaGVhZCAtYyAyNCB8fCB0cnVlKSIKICBbWyAkeyNwfSAtZXEgMjQgXV0gfHwgZGllICJmYWlsZWQgdG8gZ2VuZXJhdGUgYSBwYXNzd29yZCIKICAoIHVtYXNrIDA3NzsgcHJpbnRmICclcycgIiRwIiA+IiRmIiApCiAgY2htb2QgNjAwIC0tICIkZiIgMj4vZGV2L251bGwgfHwgdHJ1ZQogIHByaW50ZiAnJXMnICIkcCIKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gbG9ncyAtLS0tCmNzX3JvdGF0ZV9sb2coKSB7CiAgbG9jYWwgZj0kMSBtYXg9JHsyOi01MjQyODgwfSBzegogIFtbIC1mICIkZiIgXV0gfHwgcmV0dXJuIDAKICBzej0iJCh3YyAtYyA8IiRmIiAyPi9kZXYvbnVsbCB8fCBwcmludGYgMCkiCiAgaWYgKChzeiA+IG1heCkpOyB0aGVuCiAgICBtdiAtZiAtLSAiJGYiICIke2Z9LjEiCiAgZmkKICByZXR1cm4gMAp9Cgpjc19kdW1wX2xvZygpIHsKICBsb2NhbCBmPSQxIG49JHsyOi00MH0KICBbWyAtciAiJGYiIF1dIHx8IHJldHVybiAwCiAgd2FybiAibGFzdCAke259IGxpbmVzIG9mICR7Zn06IgogIHRhaWwgLW4gIiRuIiAtLSAiJGYiID4mMiB8fCB0cnVlCn0K'
    ),
    'install.sh': (
        'IyEvdXNyL2Jpbi9lbnYgYmFzaAojCiMgY29sYWIvaW5zdGFsbC5zaCDigJQgaW5zdGFsbCBjb2RlLXNlcnZlciArIGEgdHVubmVsIGFnZW50IGluc2lkZSBhIENvbGFiIFZNLgojCiMgICBiYXNoIGNvbGFiL2luc3RhbGwuc2gKIyAgIFdJVEhfQ0xPVURGTEFSRUQ9MSBiYXNoIGNvbGFiL2luc3RhbGwuc2gKIyAgIENTX1ZFUlNJT049NC4xNDUuMiBiYXNoIGNvbGFiL2luc3RhbGwuc2ggICAgICMgcGluCiMgICBDU19DQUNIRV9ESVI9L2NvbnRlbnQvZHJpdmUvTXlEcml2ZS8uY3MtY2FjaGUgYmFzaCBjb2xhYi9pbnN0YWxsLnNoCiMKIyBUaGUgcmVsZWFzZSB0YXJiYWxsIGlzIGNhY2hlZCBpbiB0aGUgc3RhdGUgZGlyLiBQb2ludCBDU19DQUNIRV9ESVIgYXQgRHJpdmUKIyBhbmQgYSBmcmVzaCBydW50aW1lIHJlaW5zdGFsbHMgaW4gc2Vjb25kcyBpbnN0ZWFkIG9mIHJlLWRvd25sb2FkaW5nIH4xMDAgTUIuCiMKc2V0IC1FZXVvIHBpcGVmYWlsCkhFUkU9IiQoY2QgLS0gIiQoZGlybmFtZSAtLSAiJHtCQVNIX1NPVVJDRVswXX0iKSIgJiYgcHdkIC1QKSIKIyBzaGVsbGNoZWNrIHNvdXJjZT1saWIvY29tbW9uLnNoCnNvdXJjZSAiJHtIRVJFfS9saWIvY29tbW9uLnNoIgoKQ1NfUFJFRklYPSIke0NTX1BSRUZJWDotL3Vzci9sb2NhbH0iCkNTX1ZFUlNJT049IiR7Q1NfVkVSU0lPTjotfSIKV0lUSF9DTE9VREZMQVJFRD0iJHtXSVRIX0NMT1VERkxBUkVEOi0wfSIKU0tJUF9BUFQ9IiR7U0tJUF9BUFQ6LTB9IgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSByZXNvbHZlIHRoZSB2ZXJzaW9uIC0tCnJlc29sdmVfdmVyc2lvbigpIHsKICBpZiBbWyAtbiAiJENTX1ZFUlNJT04iIF1dOyB0aGVuCiAgICBwcmludGYgJyVzJyAiJHtDU19WRVJTSU9OI3Z9IgogICAgcmV0dXJuIDAKICBmaQogIGxvY2FsIGluc3RhbGxlZAogIGluc3RhbGxlZD0iJChjb2RlLXNlcnZlciAtLXZlcnNpb24gMj4vZGV2L251bGwgfCBhd2sgJ05SPT0xIHsgcHJpbnQgJDEgfScgfHwgdHJ1ZSkiCiAgaWYgW1sgLW4gIiRpbnN0YWxsZWQiIF1dOyB0aGVuCiAgICBwcmludGYgJyVzJyAiJGluc3RhbGxlZCIKICAgIHJldHVybiAwCiAgZmkKICAjIEhFQUQgb24gL3JlbGVhc2VzL2xhdGVzdCBmb2xsb3dzIHRoZSByZWRpcmVjdCB0byAvcmVsZWFzZXMvdGFnL3ZYLlkuWiBhbmQKICAjIGF2b2lkcyB0aGUgdW5hdXRoZW50aWNhdGVkIEdpdEh1YiBBUEkgcmF0ZSBsaW1pdCB0aGF0IHNoYXJlZCBDb2xhYiBlZ3Jlc3MKICAjIElQcyBoaXQgY29uc3RhbnRseS4KICBsb2NhbCB0YWcKICB0YWc9IiQoY3VybCAtZnNTTCAtbyAvZGV2L251bGwgLXcgJyV7dXJsX2VmZmVjdGl2ZX0nIFwKICAgIGh0dHBzOi8vZ2l0aHViLmNvbS9jb2Rlci9jb2RlLXNlcnZlci9yZWxlYXNlcy9sYXRlc3QgMj4vZGV2L251bGwgfHwgdHJ1ZSkiCiAgdGFnPSIke3RhZyMjKi90YWcvdn0iCiAgaWYgW1sgISAiJHRhZyIgPX4gXlswLTldK1wuWzAtOV0rXC5bMC05XSskIF1dOyB0aGVuCiAgICBkaWUgImNvdWxkIG5vdCByZXNvbHZlIHRoZSBsYXRlc3QgY29kZS1zZXJ2ZXIgdmVyc2lvbjsgc2V0IENTX1ZFUlNJT04iCiAgZmkKICBwcmludGYgJyVzJyAiJHRhZyIKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gcGFja2FnZXMgLS0KaW5zdGFsbF9wYWNrYWdlcygpIHsKICBsb2NhbCBwa2dzCiAgcGtncz0iY2EtY2VydGlmaWNhdGVzIGN1cmwgd2dldCB4ei11dGlscyBsaWJuc3MzIGxpYnNlY3JldC0xLTAgbGlieGtiZmlsZTEgbGliZ2JtMSBsaWJ4c2htZmVuY2UxICQoY3NfYXNvdW5kX3BrZykiCiAgbG9nICJpbnN0YWxsaW5nIHJ1bnRpbWUgbGlicmFyaWVzOiAke3BrZ3N9IgogIGNzX3Jvb3QgYXB0LWdldCB1cGRhdGUgLXFxCiAgIyBzaGVsbGNoZWNrIGRpc2FibGU9U0MyMDg2ICAjIGRlbGliZXJhdGUgd29yZCBzcGxpdHRpbmcgb2YgdGhlIHBhY2thZ2UgbGlzdAogIERFQklBTl9GUk9OVEVORD1ub25pbnRlcmFjdGl2ZSBjc19yb290IGFwdC1nZXQgaW5zdGFsbCAteSAtcXEgLS1uby1pbnN0YWxsLXJlY29tbWVuZHMgJHBrZ3MKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY29kZS1zZXJ2ZXIgLS0KIyBPZmZpY2lhbCBzdGFuZGFsb25lIGxheW91dCwgcmVwbGljYXRlZCBzbyBgY29kZS1zZXJ2ZXJgIHJlc29sdmVzIG5vcm1hbGx5OgojICAgL3Vzci9sb2NhbC9saWIvY29kZS1zZXJ2ZXIgLT4gY29kZS1zZXJ2ZXItPHZlcj4KIyAgIC91c3IvbG9jYWwvYmluL2NvZGUtc2VydmVyIC0+IC4uL2xpYi9jb2RlLXNlcnZlci9iaW4vY29kZS1zZXJ2ZXIKaW5zdGFsbF9jb2RlX3NlcnZlcigpIHsKICBsb2NhbCB2ZXJzaW9uPSQxIGFyY2ggY2FjaGUgdGFyYmFsbCByb290IGxpbmsKCiAgYXJjaD0iJChjc19hcmNoKSIKICBjYWNoZT0iJChjc19jYWNoZV9kaXIpIgogIHRhcmJhbGw9IiR7Y2FjaGV9L2NvZGUtc2VydmVyLSR7dmVyc2lvbn0tbGludXgtJHthcmNofS50YXIuZ3oiCiAgcm9vdD0iJHtDU19QUkVGSVh9L2xpYi9jb2RlLXNlcnZlci0ke3ZlcnNpb259IgogIGxpbms9IiR7Q1NfUFJFRklYfS9saWIvY29kZS1zZXJ2ZXIiCgogIGlmIFtbIC14ICIke3Jvb3R9L2Jpbi9jb2RlLXNlcnZlciIgXV07IHRoZW4KICAgIG9rICJjb2RlLXNlcnZlciAke3ZlcnNpb259IGFscmVhZHkgdW5wYWNrZWQgYXQgJHtyb290fSIKICBlbHNlCiAgICBpZiBbWyAhIC1zICIkdGFyYmFsbCIgXV07IHRoZW4KICAgICAgbG9jYWwgdXJsPSJodHRwczovL2dpdGh1Yi5jb20vY29kZXIvY29kZS1zZXJ2ZXIvcmVsZWFzZXMvZG93bmxvYWQvdiR7dmVyc2lvbn0vY29kZS1zZXJ2ZXItJHt2ZXJzaW9ufS1saW51eC0ke2FyY2h9LnRhci5neiIKICAgICAgbG9nICJkb3dubG9hZGluZyAke3ZlcnNpb259ICgke2FyY2h9KSIKICAgICAgaWYgISBjdXJsIC1mTCAtLXJldHJ5IDMgLS1yZXRyeS1kZWxheSAyIC0jIC1vICIke3RhcmJhbGx9LnBhcnQiICIkdXJsIjsgdGhlbgogICAgICAgIHJtIC1mIC0tICIke3RhcmJhbGx9LnBhcnQiCiAgICAgICAgd2FybiAidGFyYmFsbCBkb3dubG9hZCBmYWlsZWQ7IGZhbGxpbmcgYmFjayB0byB0aGUgb2ZmaWNpYWwgaW5zdGFsbCBzY3JpcHQiCiAgICAgICAgY3VybCAtZnNTTCBodHRwczovL2NvZGUtc2VydmVyLmRldi9pbnN0YWxsLnNoIFwKICAgICAgICAgIHwgc2ggLXMgLS0gLS1tZXRob2Q9c3RhbmRhbG9uZSAtLXByZWZpeD0iJENTX1BSRUZJWCIgLS12ZXJzaW9uPSIkdmVyc2lvbiIKICAgICAgICByZXR1cm4gMAogICAgICBmaQogICAgICBtdiAtZiAtLSAiJHt0YXJiYWxsfS5wYXJ0IiAiJHRhcmJhbGwiCiAgICBlbHNlCiAgICAgIG9rICJ1c2luZyBjYWNoZWQgdGFyYmFsbCAkKGJhc2VuYW1lIC0tICIkdGFyYmFsbCIpIgogICAgZmkKCiAgICBsb2cgInVucGFja2luZyBpbnRvICR7Q1NfUFJFRklYfS9saWIiCiAgICBsb2NhbCBzdGFnZSBleHRyYWN0ZWQKICAgIHN0YWdlPSIkKG1rdGVtcCAtZCkiCiAgICB0YXIgLXh6ZiAiJHRhcmJhbGwiIC1DICIkc3RhZ2UiCiAgICBleHRyYWN0ZWQ9IiQoZmluZCAiJHN0YWdlIiAtbWF4ZGVwdGggMSAtbWluZGVwdGggMSAtdHlwZSBkIHwgaGVhZCAtMSkiCiAgICBpZiBbWyAteiAiJGV4dHJhY3RlZCIgXV07IHRoZW4KICAgICAgcm0gLXJmIC0tICIkc3RhZ2UiCiAgICAgIGRpZSAidW5leHBlY3RlZCB0YXJiYWxsIGxheW91dCBpbiAkKGJhc2VuYW1lIC0tICIkdGFyYmFsbCIpIgogICAgZmkKICAgIGNzX3Jvb3Qgcm0gLXJmIC0tICIkcm9vdCIKICAgIGNzX3Jvb3QgbXYgLS0gIiRleHRyYWN0ZWQiICIkcm9vdCIKICAgIHJtIC1yZiAtLSAiJHN0YWdlIgogIGZpCgogIGNzX3Jvb3QgbG4gLXNmbiAtLSAiY29kZS1zZXJ2ZXItJHt2ZXJzaW9ufSIgIiRsaW5rIgogIGNzX3Jvb3QgbWtkaXIgLXAgLS0gIiR7Q1NfUFJFRklYfS9iaW4iCiAgY3Nfcm9vdCBsbiAtc2ZuIC0tICIuLi9saWIvY29kZS1zZXJ2ZXIvYmluL2NvZGUtc2VydmVyIiAiJHtDU19QUkVGSVh9L2Jpbi9jb2RlLXNlcnZlciIKICBoYXNoIC1yCgogIFtbIC14ICIke0NTX1BSRUZJWH0vYmluL2NvZGUtc2VydmVyIiBdXSB8fCBkaWUgImNvZGUtc2VydmVyIG5vdCBvbiBQQVRIIGF0ICR7Q1NfUFJFRklYfS9iaW4iCiAgb2sgImNvZGUtc2VydmVyICQoY29kZS1zZXJ2ZXIgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgaGVhZCAtMSkiCn0KCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIG5ncm9rIC0tCmluc3RhbGxfbmdyb2soKSB7CiAgaWYgY29tbWFuZCAtdiBuZ3JvayA+L2Rldi9udWxsIDI+JjE7IHRoZW4KICAgIG9rICJuZ3JvayAkKG5ncm9rIC0tdmVyc2lvbiAyPi9kZXYvbnVsbCB8IGhlYWQgLTEpIgogICAgcmV0dXJuIDAKICBmaQogIGxvY2FsIGFyY2ggdG1wCiAgYXJjaD0iJChjc19hcmNoKSIKICB0bXA9IiQobWt0ZW1wIC1kKSIKICBsb2cgImluc3RhbGxpbmcgbmdyb2sgYWdlbnQgKCR7YXJjaH0pIgogIGN1cmwgLWZzU0wgLS1yZXRyeSAzIFwKICAgICJodHRwczovL2Jpbi5lcXVpbm94LmlvL2MvYk55ajFtUVZZNGMvbmdyb2stdjMtc3RhYmxlLWxpbnV4LSR7YXJjaH0udGd6IiBcCiAgICAtbyAiJHt0bXB9L25ncm9rLnRneiIgfHwgZGllICJuZ3JvayBkb3dubG9hZCBmYWlsZWQiCiAgdGFyIC14emYgIiR7dG1wfS9uZ3Jvay50Z3oiIC1DICIkdG1wIgogIGNzX3Jvb3QgaW5zdGFsbCAtbSAwNzU1ICIke3RtcH0vbmdyb2siICIke0NTX1BSRUZJWH0vYmluL25ncm9rIgogIHJtIC1yZiAtLSAiJHRtcCIKICBoYXNoIC1yCiAgb2sgIm5ncm9rICQobmdyb2sgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgaGVhZCAtMSkiCn0KCmNvbmZpZ3VyZV9uZ3JvaygpIHsKICBpZiBbWyAtbiAiJHtOR1JPS19UT0tFTjotfSIgXV07IHRoZW4KICAgIGxvZyAiY29uZmlndXJpbmcgbmdyb2sgYXV0aHRva2VuIgogICAgbmdyb2sgY29uZmlnIGFkZC1hdXRodG9rZW4gIiROR1JPS19UT0tFTiIgPi9kZXYvbnVsbCAyPiYxIFwKICAgICAgfHwgd2FybiAibmdyb2sgY29uZmlnIGFkZC1hdXRodG9rZW4gZmFpbGVkIgogICAgcmV0dXJuIDAKICBmaQogIGlmIG5ncm9rIGNvbmZpZyBjaGVjayA+L2Rldi9udWxsIDI+JjE7IHRoZW4KICAgIG9rICJuZ3JvayBhdXRodG9rZW4gYWxyZWFkeSBjb25maWd1cmVkIgogICAgcmV0dXJuIDAKICBmaQogIHdhcm4gIm5vIE5HUk9LX1RPS0VOIGFuZCBubyBhdXRodG9rZW4gY29uZmlndXJlZCDigJQgbmdyb2sgd2lsbCBmYWlsIHVudGlsIHlvdSBydW46IgogIHdhcm4gIiAgbmdyb2sgY29uZmlnIGFkZC1hdXRodG9rZW4gPHRva2VuPiAgICAjIGRhc2hib2FyZC5uZ3Jvay5jb20vZ2V0LXN0YXJ0ZWQveW91ci1hdXRodG9rZW4iCiAgcmV0dXJuIDAKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY2xvdWRmbGFyZWQgLS0KaW5zdGFsbF9jbG91ZGZsYXJlZCgpIHsKICBpZiBjb21tYW5kIC12IGNsb3VkZmxhcmVkID4vZGV2L251bGwgMj4mMTsgdGhlbgogICAgb2sgImNsb3VkZmxhcmVkICQoY2xvdWRmbGFyZWQgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgYXdrICdOUj09MSB7IHByaW50ICQzIH0nKSIKICAgIHJldHVybiAwCiAgZmkKICBsb2NhbCBhcmNoCiAgYXJjaD0iJChjc19hcmNoKSIKICBsb2cgImluc3RhbGxpbmcgY2xvdWRmbGFyZWQgKCR7YXJjaH0pIgogIGN1cmwgLWZzU0wgLS1yZXRyeSAzIFwKICAgICJodHRwczovL2dpdGh1Yi5jb20vY2xvdWRmbGFyZS9jbG91ZGZsYXJlZC9yZWxlYXNlcy9sYXRlc3QvZG93bmxvYWQvY2xvdWRmbGFyZWQtbGludXgtJHthcmNofSIgXAogICAgLW8gL3RtcC9jbG91ZGZsYXJlZC5iaW4gfHwgZGllICJjbG91ZGZsYXJlZCBkb3dubG9hZCBmYWlsZWQiCiAgY3Nfcm9vdCBpbnN0YWxsIC1tIDA3NTUgL3RtcC9jbG91ZGZsYXJlZC5iaW4gIiR7Q1NfUFJFRklYfS9iaW4vY2xvdWRmbGFyZWQiCiAgcm0gLWYgLS0gL3RtcC9jbG91ZGZsYXJlZC5iaW4KICBoYXNoIC1yCiAgb2sgImNsb3VkZmxhcmVkICQoY2xvdWRmbGFyZWQgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgYXdrICdOUj09MSB7IHByaW50ICQzIH0nKSIKfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIG1haW4gLS0KbWFpbigpIHsKICBsb2NhbCBzZAogIHNkPSIkKGNzX3N0YXRlX2RpcikiCgogIGxvY2FsIHZlcnNpb24KICB2ZXJzaW9uPSIkKHJlc29sdmVfdmVyc2lvbikiCiAgbG9nICJ0YXJnZXQgY29kZS1zZXJ2ZXIgdmVyc2lvbjogJHt2ZXJzaW9ufSIKCiAgaWYgW1sgIiRTS0lQX0FQVCIgIT0gIjEiIF1dOyB0aGVuCiAgICBpZiBjb21tYW5kIC12IGNvZGUtc2VydmVyID4vZGV2L251bGwgMj4mMSAmJiBbWyAiJChjb2RlLXNlcnZlciAtLXZlcnNpb24gMj4vZGV2L251bGwgfCBhd2sgJ05SPT0xIHsgcHJpbnQgJDEgfScpIiA9PSAiJHZlcnNpb24iIF1dOyB0aGVuCiAgICAgIG9rICJydW50aW1lIGxpYnJhcmllcyBhc3N1bWVkIHByZXNlbnQgKGNvZGUtc2VydmVyICR7dmVyc2lvbn0gYWxyZWFkeSBpbnN0YWxsZWQpIgogICAgZWxzZQogICAgICBpbnN0YWxsX3BhY2thZ2VzCiAgICBmaQogIGZpCgogIGluc3RhbGxfY29kZV9zZXJ2ZXIgIiR2ZXJzaW9uIgogIGluc3RhbGxfbmdyb2sKICBjb25maWd1cmVfbmdyb2sKICBpZiBbWyAiJFdJVEhfQ0xPVURGTEFSRUQiID09ICIxIiBdXTsgdGhlbgogICAgaW5zdGFsbF9jbG91ZGZsYXJlZAogIGZpCgogIHByaW50ZiAnJXNcbicgIiR2ZXJzaW9uIiA+IiR7c2R9Ly5pbnN0YWxsZWQtdmVyc2lvbiIKCiAgY2F0IDw8RU9GCgogIGluc3RhbGwgY29tcGxldGUKCiAgbmV4dCAgICAgYmFzaCBjb2xhYi9zZXJ2ZS5zaAogIG9yICAgICAgIGJhc2ggY29sYWIvc3VwZXJ2aXNlLnNoICAgICAgKHJlc3RhcnRzICsgaWRsZS9tYXgtcnVudGltZSB3YXRjaGRvZykKICBjaGVjayAgICBiYXNoIGNvbGFiL2RvY3Rvci5zaAoKRU9GCn0KCm1haW4gIiRAIgo='
    ),
    'serve.sh': (
        'IyEvdXNyL2Jpbi9lbnYgYmFzaAojCiMgY29sYWIvc2VydmUuc2gg4oCUIHN0YXJ0IGNvZGUtc2VydmVyIGFuZCBleHBvc2UgaXQgdGhyb3VnaCBhIHR1bm5lbC4KIwojICAgYmFzaCBjb2xhYi9zZXJ2ZS5zaCAgICAgICAgICAgICAgICAgICAgICAjIG5ncm9rLCBiYWNrZ3JvdW5kLCBwcmludHMgVVJMCiMgICBiYXNoIGNvbGFiL3NlcnZlLnNoIC0tZm9yZWdyb3VuZCAgICAgICAgICMgYmxvY2sgKHVzZWQgYnkgc3VwZXJ2aXNlLnNoKQojICAgQ1NfVFVOTkVMPWNsb3VkZmxhcmVkIGJhc2ggY29sYWIvc2VydmUuc2gKIyAgIE5HUk9LX0RPTUFJTj1teS1kZXYubmdyb2stZnJlZS5hcHAgYmFzaCBjb2xhYi9zZXJ2ZS5zaCAgICMgc3RhYmxlIGhvc3RuYW1lCiMKIyBOb3RoaW5nIGhlcmUgbW9kaWZpZXMgY29kZS1zZXJ2ZXIuIEl0IGlzIGEgcGxhaW4gTm9kZSBIVFRQIHNlcnZlcjsgdGhpcyBzY3JpcHQKIyBvbmx5IHBpY2tzIGZsYWdzIHRoYXQgc3VpdCBhbiBlcGhlbWVyYWwgQ29sYWIgVk0uCiMKc2V0IC1FZXVvIHBpcGVmYWlsCkhFUkU9IiQoY2QgLS0gIiQoZGlybmFtZSAtLSAiJHtCQVNIX1NPVVJDRVswXX0iKSIgJiYgcHdkIC1QKSIKIyBzaGVsbGNoZWNrIHNvdXJjZT1saWIvY29tbW9uLnNoCnNvdXJjZSAiJHtIRVJFfS9saWIvY29tbW9uLnNoIgoKRk9SRUdST1VORD0wClRVTk5FTD0iJHtDU19UVU5ORUw6LWF1dG99IgpQT1JUPSIke0NTX1BPUlQ6LTgwODB9IgpBVVRIPSJwYXNzd29yZCIKCnVzYWdlKCkgewogIGNhdCA8PCdFT0YnCnVzYWdlOiBzZXJ2ZS5zaCBbb3B0aW9uc10KCiAgLS1mb3JlZ3JvdW5kICAgICByZW1haW4gaW4gdGhlIGZvcmVncm91bmQgdW50aWwgYSBjaGlsZCBleGl0cwogIC0tdHVubmVsPU5BTUUgICAgYXV0byB8IG5ncm9rIHwgY2xvdWRmbGFyZWQgfCBub25lIChkZWZhdWx0OiAkQ1NfVFVOTkVMLCBhdXRvKQogIC0tcG9ydD1OICAgICAgICAgbG9jYWwgcG9ydCAgICAgICAgICAgICAgICAgICAgICAgIChkZWZhdWx0OiAkQ1NfUE9SVCwgODA4MCkKICAtLW5vLWF1dGggICAgICAgIGRpc2FibGUgYXV0aGVudGljYXRpb24gKHJlcXVpcmVzIENTX0FMTE9XX0lOU0VDVVJFPTEpCiAgLWgsIC0taGVscApFT0YKfQoKd2hpbGUgKCgkIykpOyBkbwogIGNhc2UgIiQxIiBpbgogICAgLS1mb3JlZ3JvdW5kKSBGT1JFR1JPVU5EPTEgOzsKICAgIC0tdHVubmVsPSopICAgVFVOTkVMPSIkezEjKj19IiA7OwogICAgLS1wb3J0PSopICAgICBQT1JUPSIkezEjKj19IiA7OwogICAgLS1uby1hdXRoKSAgICBBVVRIPSJub25lIiA7OwogICAgLWh8LS1oZWxwKSAgICB1c2FnZTsgZXhpdCAwIDs7CiAgICAqKSAgICAgICAgICAgIGRpZSAidW5rbm93biBhcmd1bWVudDogJDEgKHRyeSAtLWhlbHApIiA7OwogIGVzYWMKICBzaGlmdApkb25lCgpbWyAiJFBPUlQiID1+IF5bMC05XSskIF1dIHx8IGRpZSAiaW52YWxpZCBwb3J0OiAke1BPUlR9IgpjYXNlICIkVFVOTkVMIiBpbgogIGF1dG98bmdyb2t8Y2xvdWRmbGFyZWR8bm9uZSkgOzsKICAqKSBkaWUgIi0tdHVubmVsIG11c3QgYmUgYXV0bywgbmdyb2ssIGNsb3VkZmxhcmVkIG9yIG5vbmUgKGdvdDogJHtUVU5ORUx9KSIgOzsKZXNhYwoKIyBhdXRvOiBuZ3JvayBpZiBpdCBoYXMgY3JlZGVudGlhbHMsIG90aGVyd2lzZSBjbG91ZGZsYXJlZCBxdWljayB0dW5uZWwsIHdoaWNoCiMgbmVlZHMgbm8gYWNjb3VudCBhdCBhbGwuIFRoYXQgbWFrZXMgYSBiYXJlIG5vdGVib29rIHJ1biB3b3JrIHdpdGggemVybyBzZXR1cC4KaWYgW1sgIiRUVU5ORUwiID09ICJhdXRvIiBdXTsgdGhlbgogIGlmIFtbIC1uICIke05HUk9LX1RPS0VOOi19IiBdXTsgdGhlbgogICAgVFVOTkVMPW5ncm9rCiAgZWxpZiBjb21tYW5kIC12IG5ncm9rID4vZGV2L251bGwgMj4mMSAmJiBuZ3JvayBjb25maWcgY2hlY2sgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICBUVU5ORUw9bmdyb2sKICBlbGlmIGNvbW1hbmQgLXYgY2xvdWRmbGFyZWQgPi9kZXYvbnVsbCAyPiYxOyB0aGVuCiAgICBUVU5ORUw9Y2xvdWRmbGFyZWQKICBlbHNlCiAgICBUVU5ORUw9bmdyb2sKICBmaQogIGxvZyAiYXV0by1zZWxlY3RlZCB0dW5uZWw6ICR7VFVOTkVMfSIKZmkKCmlmIFtbICIkQVVUSCIgPT0gIm5vbmUiIF1dOyB0aGVuCiAgW1sgIiR7Q1NfQUxMT1dfSU5TRUNVUkU6LTB9IiA9PSAiMSIgXV0gXAogICAgfHwgZGllICItLW5vLWF1dGggcmVxdWlyZXMgQ1NfQUxMT1dfSU5TRUNVUkU9MSIKICBbWyAiJFRVTk5FTCIgPT0gIm5vbmUiIF1dIFwKICAgIHx8IGRpZSAiLS1uby1hdXRoIGNhbm5vdCBiZSBjb21iaW5lZCB3aXRoIGEgcHVibGljIHR1bm5lbCIKZmkKCmNzX25lZWQgY29kZS1zZXJ2ZXIgInJ1bjogYmFzaCBjb2xhYi9pbnN0YWxsLnNoIgpjc19uZWVkIGN1cmwKY3NfbmVlZCBweXRob24zCgpTRD0iJChjc19zdGF0ZV9kaXIpIgpjc19sb2NrIHNlcnZlCgpDU19QSUQ9JycKVFVOX1BJRD0nJwpQVUJMSUNfVVJMPScnCgpjbGVhbnVwX2NoaWxkcmVuKCkgewogIFtbIC1uICIkQ1NfUElEIiBdXSAmJiBjc19zdG9wX3BpZGZpbGUgIiR7U0R9L2NzLnBpZCIgY29kZS1zZXJ2ZXIKICBbWyAtbiAiJFRVTl9QSUQiIF1dICYmIGNzX3N0b3BfcGlkZmlsZSAiJHtTRH0vJHtUVU5ORUx9LnBpZCIgIiRUVU5ORUwiCiAgcmV0dXJuIDAKfQoKb25fZXhpdCgpIHsKICBpZiAoKEZPUkVHUk9VTkQpKTsgdGhlbiBjbGVhbnVwX2NoaWxkcmVuOyBmaQogIGNzX3VubG9jawp9CnRyYXAgb25fZXhpdCBFWElUCiMgVHVybiBzaWduYWxzIGludG8gYSBub3JtYWwgZXhpdCBzbyB0aGUgRVhJVCB0cmFwIHRlYXJzIHRoZSBjaGlsZHJlbiBkb3duLgp0cmFwICdleGl0IDE0MycgVEVSTSBJTlQgSFVQCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gc3RhbGUgaW5zdGFuY2VzIC0tLS0KbG9nICJjbGVhcmluZyBzdGFsZSBpbnN0YW5jZXMiCmNzX3N0b3BfcGlkZmlsZSAiJHtTRH0vY3MucGlkIiBjb2RlLXNlcnZlcgpjc19zdG9wX3BpZGZpbGUgIiR7U0R9L25ncm9rLnBpZCIgbmdyb2sKY3Nfc3RvcF9waWRmaWxlICIke1NEfS9jbG91ZGZsYXJlZC5waWQiIGNsb3VkZmxhcmVkCgppZiBjc19wb3J0X2luX3VzZSAiJFBPUlQiOyB0aGVuCiAgbG9nICJwb3J0ICR7UE9SVH0gaXMgb2NjdXBpZWQ7IGF0dGVtcHRpbmcgdG8gcmVjbGFpbSBpdCIKICBpZiAhIGNzX2ZyZWVfcG9ydCAiJFBPUlQiOyB0aGVuCiAgICBjc19wb3J0X3JlcG9ydCAiJFBPUlQiCiAgICBwcmludGYgJyVzZXJyJXMgcG9ydCAlcyBpcyBidXN5IGFuZCBub3QgcmVjbGFpbWFibGUg4oCUIGZyZWUgaXQgb3IgdXNlIC0tcG9ydD08bj5cbicgXAogICAgICAiJENTX0NfUkVEIiAiJENTX0NfUkVTRVQiICIkUE9SVCIgPiYyCiAgICAjIEV4aXQgY29kZSAyIG1lYW5zICJyZXRyeWluZyB3aWxsIG5vdCBoZWxwIjsgc3VwZXJ2aXNlLnNoIHN0b3BzIG9uIGl0LgogICAgZXhpdCAyCiAgZmkKICBvayAicG9ydCAke1BPUlR9IHJlY2xhaW1lZCIKZmkKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGV4dGVuc2lvbnMgLS0tLQojIEluc3RhbGxlZCBiZWZvcmUgdGhlIHNlcnZlciBzdGFydHMgc28gdGhlIG1hcmtlciBmaWxlIGFuZCB0aGUgZXh0ZW5zaW9uIGRpcgojIGNhbm5vdCByYWNlIHdpdGggdGhlIGV4dGVuc2lvbiBob3N0LgppbnN0YWxsX2V4dGVuc2lvbnMoKSB7CiAgbG9jYWwgc2Q9JDEgbWFya2VyIGV4dAogIG1hcmtlcj0iJHtzZH0vLmV4dGVuc2lvbnMtJChwcmludGYgJyVzJyAiJENTX0VYVEVOU0lPTlMiIHwgY2tzdW0gfCBjdXQgLWQnICcgLWYxKSIKICBpZiBbWyAtZiAiJG1hcmtlciIgXV07IHRoZW4KICAgIG9rICJleHRlbnNpb25zIGFscmVhZHkgaW5zdGFsbGVkIgogICAgcmV0dXJuIDAKICBmaQogIGxvY2FsIElGUz0nLCcKICBmb3IgZXh0IGluICRDU19FWFRFTlNJT05TOyBkbwogICAgW1sgLW4gIiRleHQiIF1dIHx8IGNvbnRpbnVlCiAgICBsb2cgImluc3RhbGxpbmcgZXh0ZW5zaW9uICR7ZXh0fSIKICAgIGlmIGNvZGUtc2VydmVyIFwKICAgICAgICAtLXVzZXItZGF0YS1kaXIgIiR7c2R9L2RhdGEiIFwKICAgICAgICAtLWV4dGVuc2lvbnMtZGlyICIke3NkfS9leHRlbnNpb25zIiBcCiAgICAgICAgLS1pbnN0YWxsLWV4dGVuc2lvbiAiJGV4dCIgLS1mb3JjZSBcCiAgICAgICAgPj4iJHtzZH0vZXh0ZW5zaW9ucy5sb2ciIDI+JjE7IHRoZW4KICAgICAgb2sgImV4dGVuc2lvbiAke2V4dH0iCiAgICBlbHNlCiAgICAgIHdhcm4gImV4dGVuc2lvbiAke2V4dH0gZmFpbGVkIChzZWUgJHtzZH0vZXh0ZW5zaW9ucy5sb2cpIgogICAgZmkKICBkb25lCiAgOiA+IiRtYXJrZXIiCn0KaWYgW1sgLW4gIiR7Q1NfRVhURU5TSU9OUzotfSIgXV07IHRoZW4KICBpbnN0YWxsX2V4dGVuc2lvbnMgIiRTRCIKZmkKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gY29kZS1zZXJ2ZXIgLS0tLQpDT0RFX1BBU1NXT1JEPSIkKGNzX3Bhc3N3b3JkICIkU0QiKSIKCmNzX2FyZ3M9KAogIC0tYmluZC1hZGRyICIxMjcuMC4wLjE6JHtQT1JUfSIKICAtLWF1dGggIiRBVVRIIgogIC0tdXNlci1kYXRhLWRpciAiJHtTRH0vZGF0YSIKICAtLWV4dGVuc2lvbnMtZGlyICIke1NEfS9leHRlbnNpb25zIgogIC0tZGlzYWJsZS10ZWxlbWV0cnkKICAtLWRpc2FibGUtdXBkYXRlLWNoZWNrCiAgLS1kaXNhYmxlLXdvcmtzcGFjZS10cnVzdAopCgpsb2cgInN0YXJ0aW5nIGNvZGUtc2VydmVyIG9uIDEyNy4wLjAuMToke1BPUlR9Igpjc19yb3RhdGVfbG9nICIke1NEfS9jb2RlLXNlcnZlci5sb2ciClBBU1NXT1JEPSIkQ09ERV9QQVNTV09SRCIgbm9odXAgY29kZS1zZXJ2ZXIgIiR7Y3NfYXJnc1tAXX0iIFwKICA+IiR7U0R9L2NvZGUtc2VydmVyLmxvZyIgMj4mMSA5PiYtICYKQ1NfUElEPSQhCmNzX3dyaXRlX3BpZCAiJHtTRH0vY3MucGlkIiAiJENTX1BJRCIKCmlmICEgY3Nfd2FpdF9odHRwICJodHRwOi8vMTI3LjAuMC4xOiR7UE9SVH0vaGVhbHRoeiIgIiR7Q1NfSEVBTFRIX1RJTUVPVVQ6LTkwfSI7IHRoZW4KICBjc19kdW1wX2xvZyAiJHtTRH0vY29kZS1zZXJ2ZXIubG9nIiA2MAogIGNzX3N0b3BfcGlkZmlsZSAiJHtTRH0vY3MucGlkIiBjb2RlLXNlcnZlcgogIENTX1BJRD0nJwogIGRpZSAiY29kZS1zZXJ2ZXIgZGlkIG5vdCBiZWNvbWUgaGVhbHRoeSIKZmkKb2sgImNvZGUtc2VydmVyIGhlYWx0aHkgKHBpZCAke0NTX1BJRH0pIgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHR1bm5lbHMgLS0tLQpuZ3Jva19wdWJsaWNfdXJsKCkgewogIGN1cmwgLWZzUyAtLW1heC10aW1lIDIgLS1ub3Byb3h5ICcqJyAiaHR0cDovLzEyNy4wLjAuMTo0MDQwL2FwaS90dW5uZWxzIiAyPi9kZXYvbnVsbCBcCiAgICB8IHB5dGhvbjMgLWMgJwppbXBvcnQganNvbiwgc3lzCnRyeToKICAgIGRhdGEgPSBqc29uLmxvYWQoc3lzLnN0ZGluKQpleGNlcHQgRXhjZXB0aW9uOgogICAgc3lzLmV4aXQoMCkKZm9yIHQgaW4gZGF0YS5nZXQoInR1bm5lbHMiLCBbXSk6CiAgICB1cmwgPSB0LmdldCgicHVibGljX3VybCIsICIiKQogICAgaWYgdXJsLnN0YXJ0c3dpdGgoImh0dHBzOi8vIik6CiAgICAgICAgcHJpbnQodXJsKQogICAgICAgIGJyZWFrCicgMj4vZGV2L251bGwgfHwgdHJ1ZQp9CgpzdGFydF9uZ3JvaygpIHsKICBjc19uZWVkIG5ncm9rICJydW46IGJhc2ggY29sYWIvaW5zdGFsbC5zaCIKCiAgaWYgW1sgLW4gIiR7TkdST0tfVE9LRU46LX0iIF1dOyB0aGVuCiAgICBuZ3JvayBjb25maWcgYWRkLWF1dGh0b2tlbiAiJE5HUk9LX1RPS0VOIiA+L2Rldi9udWxsIDI+JjEgXAogICAgICB8fCB3YXJuICJuZ3JvayBhdXRodG9rZW4gcmVmcmVzaCBmYWlsZWQiCiAgZmkKCiAgbG9jYWwgYXJncz0oaHR0cCkKICBpZiBbWyAtbiAiJHtOR1JPS19ET01BSU46LX0iIF1dOyB0aGVuCiAgICBhcmdzKz0oIi0tdXJsPWh0dHBzOi8vJHtOR1JPS19ET01BSU59IikKICBmaQogIGlmIFtbIC1uICIke05HUk9LX0JBU0lDX0FVVEg6LX0iIF1dOyB0aGVuCiAgICAjIEV4dHJhIGdhdGUgYXQgbmdyb2sncyBlZGdlOiByZWplY3RzIHNjYW5uZXJzIGJlZm9yZSB0aGV5IGV2ZXIgcmVhY2gKICAgICMgY29kZS1zZXJ2ZXIuIEZyZWUgdGllciBob3N0bmFtZXMgZ2V0IHByb2JlZCB3aXRoaW4gbWludXRlcy4KICAgIGFyZ3MrPSgiLS1iYXNpYy1hdXRoPSR7TkdST0tfQkFTSUNfQVVUSH0iKQogIGZpCiAgYXJncys9KCIkUE9SVCIgLS1sb2c9c3Rkb3V0IC0tbG9nLWZvcm1hdD1qc29uKQoKICBsb2cgIm9wZW5pbmcgbmdyb2sgdHVubmVsIgogIGNzX3JvdGF0ZV9sb2cgIiR7U0R9L25ncm9rLmxvZyIKICBub2h1cCBuZ3JvayAiJHthcmdzW0BdfSIgPiIke1NEfS9uZ3Jvay5sb2ciIDI+JjEgOT4mLSAmCiAgVFVOX1BJRD0kIQogIGNzX3dyaXRlX3BpZCAiJHtTRH0vbmdyb2sucGlkIiAiJFRVTl9QSUQiCgogIGxvY2FsIGkKICBmb3IgKChpID0gMDsgaSA8IDQ1OyBpKyspKTsgZG8KICAgIFBVQkxJQ19VUkw9IiQobmdyb2tfcHVibGljX3VybCkiCiAgICBbWyAtbiAiJFBVQkxJQ19VUkwiIF1dICYmIHJldHVybiAwCiAgICBraWxsIC0wICIkVFVOX1BJRCIgMj4vZGV2L251bGwgfHwgYnJlYWsKICAgIGlmICgoaSA+IDAgJiYgaSAlIDEwID09IDApKTsgdGhlbiBkaW0gIiAgd2FpdGluZyBmb3Igbmdyb2sgVVJMICgke2l9cykiOyBmaQogICAgc2xlZXAgMQogIGRvbmUKCiAgY3NfZHVtcF9sb2cgIiR7U0R9L25ncm9rLmxvZyIgNDAKICB3YXJuICJubyBwdWJsaWMgVVJMIGZyb20gbmdyb2siCiAgd2FybiAiY29tbW9uIGNhdXNlczogbWlzc2luZyBhdXRodG9rZW4gKG5ncm9rIGNvbmZpZyBhZGQtYXV0aHRva2VuIDx0PiksIgogIHdhcm4gImZyZWUtdGllciBzZXNzaW9uIGxpbWl0IChvbmUgYWdlbnQgYXQgYSB0aW1lKSwgb3IgYSByZXNlcnZlZC1kb21haW4gbWlzbWF0Y2giCiAgcmV0dXJuIDEKfQoKc3RhcnRfY2xvdWRmbGFyZWQoKSB7CiAgY3NfbmVlZCBjbG91ZGZsYXJlZCAicnVuOiBXSVRIX0NMT1VERkxBUkVEPTEgYmFzaCBjb2xhYi9pbnN0YWxsLnNoIgoKICBsb2NhbCBhcmdzPSh0dW5uZWwpCiAgaWYgW1sgLW4gIiR7Q0ZfVE9LRU46LX0iIF1dOyB0aGVuCiAgICAjIE5hbWVkIHR1bm5lbDogc3RhYmxlIGhvc3RuYW1lIG9uIGEgZG9tYWluIHlvdSBjb250cm9sLCBubyBpbnRlcnN0aXRpYWwKICAgICMgcGFnZSwgc3Vydml2ZXMgcmVzdGFydHMuIFRoaXMgaXMgdGhlIGFuc3dlciB0byAidGhlIG5ncm9rIFVSTCBrZWVwcwogICAgIyBjaGFuZ2luZyIg4oCUIHByb3BlciBob3N0aW5nIHJhdGhlciB0aGFuIGEgZGlzZ3Vpc2UuCiAgICBhcmdzKz0ocnVuIC0tdG9rZW4gIiRDRl9UT0tFTiIpCiAgZWxzZQogICAgYXJncys9KC0tdXJsICJodHRwOi8vMTI3LjAuMC4xOiR7UE9SVH0iKQogIGZpCiAgYXJncys9KC0tbm8tYXV0b3VwZGF0ZSkKCiAgbG9nICJvcGVuaW5nIGNsb3VkZmxhcmVkIHR1bm5lbCIKICBjc19yb3RhdGVfbG9nICIke1NEfS9jbG91ZGZsYXJlZC5sb2ciCiAgbm9odXAgY2xvdWRmbGFyZWQgIiR7YXJnc1tAXX0iID4iJHtTRH0vY2xvdWRmbGFyZWQubG9nIiAyPiYxIDk+Ji0gJgogIFRVTl9QSUQ9JCEKICBjc193cml0ZV9waWQgIiR7U0R9L2Nsb3VkZmxhcmVkLnBpZCIgIiRUVU5fUElEIgoKICBpZiBbWyAtbiAiJHtDRl9UVU5ORUxfVVJMOi19IiBdXTsgdGhlbgogICAgUFVCTElDX1VSTD0iJENGX1RVTk5FTF9VUkwiCiAgICByZXR1cm4gMAogIGZpCgogIGxvY2FsIGkKICBmb3IgKChpID0gMDsgaSA8IDQ1OyBpKyspKTsgZG8KICAgIFBVQkxJQ19VUkw9IiQoZ3JlcCAtb0UgJ2h0dHBzOi8vW2EtejAtOS1dK1wudHJ5Y2xvdWRmbGFyZVwuY29tJyBcCiAgICAgICIke1NEfS9jbG91ZGZsYXJlZC5sb2ciIDI+L2Rldi9udWxsIHwgaGVhZCAtMSB8fCB0cnVlKSIKICAgIFtbIC1uICIkUFVCTElDX1VSTCIgXV0gJiYgcmV0dXJuIDAKICAgIGtpbGwgLTAgIiRUVU5fUElEIiAyPi9kZXYvbnVsbCB8fCBicmVhawogICAgaWYgKChpID4gMCAmJiBpICUgMTAgPT0gMCkpOyB0aGVuIGRpbSAiICB3YWl0aW5nIGZvciBjbG91ZGZsYXJlZCBVUkwgKCR7aX1zKSI7IGZpCiAgICBzbGVlcCAxCiAgZG9uZQoKICBjc19kdW1wX2xvZyAiJHtTRH0vY2xvdWRmbGFyZWQubG9nIiA0MAogIHdhcm4gIm5vIHB1YmxpYyBVUkwgZnJvbSBjbG91ZGZsYXJlZCIKICB3YXJuICJ3aXRoIENGX1RPS0VOIHNldCwgcGFzcyBDRl9UVU5ORUxfVVJMPWh0dHBzOi8veW91ci5ob3N0bmFtZSBzbyBpdCBjYW4gYmUgcmVwb3J0ZWQiCiAgcmV0dXJuIDEKfQoKY2FzZSAiJFRVTk5FTCIgaW4KICBuZ3JvaykgICAgICAgc3RhcnRfbmdyb2sgICAgICAgfHwgZGllICJ0dW5uZWwgc3RhcnR1cCBmYWlsZWQiIDs7CiAgY2xvdWRmbGFyZWQpIHN0YXJ0X2Nsb3VkZmxhcmVkIHx8IGRpZSAidHVubmVsIHN0YXJ0dXAgZmFpbGVkIiA7OwogIG5vbmUpICAgICAgICBQVUJMSUNfVVJMPSJodHRwOi8vMTI3LjAuMC4xOiR7UE9SVH0iIDs7CmVzYWMKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSByZXBvcnQgLS0tLQp7CiAgcHJpbnRmICd1cmw9JXNcbicgIiRQVUJMSUNfVVJMIgogIHByaW50ZiAncGFzc3dvcmQ9JXNcbicgIiRDT0RFX1BBU1NXT1JEIgogIHByaW50ZiAndHVubmVsPSVzXG4nICIkVFVOTkVMIgogIHByaW50ZiAncG9ydD0lc1xuJyAiJFBPUlQiCiAgcHJpbnRmICdzdGF0ZT0lc1xuJyAiJFNEIgp9ID4iJHtTRH0vY3JlZGVudGlhbHMudHh0IgpjaG1vZCA2MDAgLS0gIiR7U0R9L2NyZWRlbnRpYWxzLnR4dCIgMj4vZGV2L251bGwgfHwgdHJ1ZQoKcHJpbnRmICdcbicKZGltICAnICDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAnCnByaW50ZiAnICAgJXMgICVzXG4nICIkKHByaW50ZiAnXDAzM1sxbWNvZGUtc2VydmVyXDAzM1swbScpIiAiJFBVQkxJQ19VUkwiCnByaW50ZiAnICAgcGFzc3dvcmQgICAgICVzXG4nICIkQ09ERV9QQVNTV09SRCIKW1sgLW4gIiR7TkdST0tfQkFTSUNfQVVUSDotfSIgXV0gJiYgcHJpbnRmICcgICBiYXNpYyBhdXRoICAgJXNcbicgIiROR1JPS19CQVNJQ19BVVRIIgpkaW0gICcgIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgCcKcHJpbnRmICdcbicKZGltICIgIHN0YXRlICAgJHtTRH0iCmRpbSAiICBsb2dzICAgICR7U0R9L2NvZGUtc2VydmVyLmxvZyAgJHtTRH0vJHtUVU5ORUx9LmxvZyIKZGltICIgIHN0b3AgICAgYmFzaCBjb2xhYi9zdG9wLnNoIgpwcmludGYgJ1xuJwoKaWYgW1sgIiRUVU5ORUwiID09ICJuZ3JvayIgJiYgLXogIiR7TkdST0tfRE9NQUlOOi19IiBdXTsgdGhlbgogIHdhcm4gIm5ncm9rIGZyZWUgdGllciBzaG93cyBhIHdhcm5pbmcgcGFnZSBvbiB0aGUgZmlyc3QgYnJvd3NlciBoaXQuIgogIHdhcm4gIkNsaWNrIHRocm91Z2ggaXQsIG9yIHRoZSBlZGl0b3IncyBKUyBhc3NldHMgY29tZSBiYWNrIGFzIEhUTUwgKGJsYW5rIHBhZ2UpLiIKICB3YXJuICJTZXQgTkdST0tfRE9NQUlOPTxyZXNlcnZlZC1kb21haW4+IHRvIGtlZXAgb25lIGhvc3RuYW1lIGFjcm9zcyByZXN0YXJ0cywiCiAgd2FybiAib3IgQ1NfVFVOTkVMPWNsb3VkZmxhcmVkIGZvciBubyBpbnRlcnN0aXRpYWwgYXQgYWxsLiIKZmkKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGZvcmVncm91bmQgLS0tLQppZiAoKEZPUkVHUk9VTkQpKTsgdGhlbgogIGxvZyAiZm9yZWdyb3VuZCBtb2RlOyBDdHJsLUMgdG8gc3RvcCIKICB3aGlsZSA6OyBkbwogICAgaWYgISBraWxsIC0wICIkQ1NfUElEIiAyPi9kZXYvbnVsbDsgdGhlbgogICAgICB3YXJuICJjb2RlLXNlcnZlciBleGl0ZWQiCiAgICAgIGV4aXQgMQogICAgZmkKICAgIGlmIFtbIC1uICIkVFVOX1BJRCIgXV0gJiYgISBraWxsIC0wICIkVFVOX1BJRCIgMj4vZGV2L251bGw7IHRoZW4KICAgICAgd2FybiAidHVubmVsIGV4aXRlZCIKICAgICAgZXhpdCAxCiAgICBmaQogICAgc2xlZXAgMgogIGRvbmUKZmkK'
    ),
    'supervise.sh': (
        'IyEvdXNyL2Jpbi9lbnYgYmFzaAojCiMgY29sYWIvc3VwZXJ2aXNlLnNoIOKAlCBrZWVwIHRoZSBzdGFjayBhbGl2ZSBhbmQgc3RvcCBpdCBiZWZvcmUgQ29sYWIgZG9lcy4KIwojICAgYmFzaCBjb2xhYi9zdXBlcnZpc2Uuc2gKIyAgIENTX0lETEVfVElNRU9VVD05MDAgQ1NfTUFYX1JVTlRJTUU9MjE2MDAgYmFzaCBjb2xhYi9zdXBlcnZpc2Uuc2gKIwojIFR3byBqb2JzOgojICAgMS4gcmVzdGFydCBzZXJ2ZS5zaCB3aXRoIGV4cG9uZW50aWFsIGJhY2tvZmYgaWYgY29kZS1zZXJ2ZXIgb3IgdGhlIHR1bm5lbCBkaWVzCiMgICAyLiBzaHV0IHRoZSBzdGFjayBkb3duIHdoZW4gbm9ib2R5IGlzIGNvbm5lY3RlZCwgb3Igd2hlbiB0aGUgcnVudGltZSBoYXMgYmVlbgojICAgICAgaGVsZCBmb3IgdG9vIGxvbmcKIwojICgyKSBpcyB0aGUgaG9uZXN0IGhhbGYgb2YgImxlc3Mgbm90aWNlYWJsZSI6IGEgR1BVIFZNIGhlbGQgb3BlbiBmb3IgaG91cnMgd2l0aAojIG5vIGludGVyYWN0aXZlIHRyYWZmaWMgaXMgdGhlIHBhdHRlcm4gdGhhdCBnZXRzIGFjY291bnRzIHRocm90dGxlZC4gUmVsZWFzaW5nCiMgaXQgd2hlbiBpZGxlIGlzIGJvdGggYmV0dGVyIG1hbm5lcnMgYW5kIGZld2VyIHNlc3Npb25zIHRvIGJlIGZsYWdnZWQuCiMKc2V0IC1FZXVvIHBpcGVmYWlsCkhFUkU9IiQoY2QgLS0gIiQoZGlybmFtZSAtLSAiJHtCQVNIX1NPVVJDRVswXX0iKSIgJiYgcHdkIC1QKSIKIyBzaGVsbGNoZWNrIHNvdXJjZT1saWIvY29tbW9uLnNoCnNvdXJjZSAiJHtIRVJFfS9saWIvY29tbW9uLnNoIgoKUE9SVD0iJHtDU19QT1JUOi04MDgwfSIKSURMRV9USU1FT1VUPSIke0NTX0lETEVfVElNRU9VVDotMTgwMH0iICAgICAjIDMwIG1pbiB3aXRoIHplcm8gY29ubmVjdGlvbnMKTUFYX1JVTlRJTUU9IiR7Q1NfTUFYX1JVTlRJTUU6LTM5NjAwfSIgICAgICAjIDExIGgsIHVuZGVyIENvbGFiJ3MgaGFyZCBjYXAKSEVBTFRIWV9BRlRFUj0iJHtDU19IRUFMVEhZX0FGVEVSOi02MDB9IiAgICAjIHVwdGltZSB0aGF0IHJlc2V0cyB0aGUgYmFja29mZgpNQVhfQkFDS09GRj0iJHtDU19NQVhfQkFDS09GRjotNjB9IgpNQVhfUkVTVEFSVFM9IiR7Q1NfTUFYX1JFU1RBUlRTOi0yMH0iCgpbWyAiJElETEVfVElNRU9VVCIgPX4gXlswLTldKyQgXV0gfHwgZGllICJDU19JRExFX1RJTUVPVVQgbXVzdCBiZSBhIG51bWJlciBvZiBzZWNvbmRzIgpbWyAiJE1BWF9SVU5USU1FIiA9fiBeWzAtOV0rJCBdXSAgfHwgZGllICJDU19NQVhfUlVOVElNRSBtdXN0IGJlIGEgbnVtYmVyIG9mIHNlY29uZHMiCgpjc19uZWVkIGNvZGUtc2VydmVyICJydW46IGJhc2ggY29sYWIvaW5zdGFsbC5zaCIKY3NfbmVlZCBjdXJsCgpTRD0iJChjc19zdGF0ZV9kaXIpIgpjc19sb2NrIHN1cGVydmlzZQp0cmFwICdjc191bmxvY2snIEVYSVQKCnN0YWNrX2FsaXZlKCkgewogIGxvY2FsIGNzX3BpZAogIGNzX3BpZD0iJChjc19yZWFkX3BpZCAiJHtTRH0vY3MucGlkIikiCiAgW1sgLW4gIiRjc19waWQiIF1dICYmIGtpbGwgLTAgIiRjc19waWQiIDI+L2Rldi9udWxsCn0KCiMgUmV0dXJuczogMCA9IHN0aWxsIHJ1bm5pbmcsIDEwID0gaWRsZSBzaHV0ZG93biwgMTEgPSBtYXggcnVudGltZSByZWFjaGVkLAojICAgICAgICAgIDEgPSBzb21ldGhpbmcgZGllZCAoY2FsbGVyIHJlc3RhcnRzKQp3YXRjaF9zdGFjaygpIHsKICBsb2NhbCBzdGFydGVkPSRTRUNPTkRTIGxhc3RfYWN0aXZlPSRTRUNPTkRTIGNvbm5zCiAgd2hpbGUgc3RhY2tfYWxpdmU7IGRvCiAgICBzbGVlcCA1CiAgICBjb25ucz0iJChjc19hY3RpdmVfY29ubnMgIiRQT1JUIikiCiAgICBpZiBbWyAiJHtjb25uczotMH0iIC1ndCAwIF1dOyB0aGVuCiAgICAgIGxhc3RfYWN0aXZlPSRTRUNPTkRTCiAgICBmaQogICAgaWYgKChTRUNPTkRTIC0gbGFzdF9hY3RpdmUgPiBJRExFX1RJTUVPVVQpKTsgdGhlbgogICAgICBsb2cgIm5vIGNvbm5lY3Rpb25zIGZvciAke0lETEVfVElNRU9VVH1zIOKAlCBzaHV0dGluZyBkb3duIgogICAgICBDU19GUk9NX1NVUEVSVklTT1I9MSBiYXNoICIke0hFUkV9L3N0b3Auc2giIHx8IHRydWUKICAgICAgcmV0dXJuIDEwCiAgICBmaQogICAgaWYgKChTRUNPTkRTIC0gc3RhcnRlZCA+IE1BWF9SVU5USU1FKSk7IHRoZW4KICAgICAgbG9nICJtYXggcnVudGltZSAke01BWF9SVU5USU1FfXMgcmVhY2hlZCDigJQgc2h1dHRpbmcgZG93biIKICAgICAgQ1NfRlJPTV9TVVBFUlZJU09SPTEgYmFzaCAiJHtIRVJFfS9zdG9wLnNoIiB8fCB0cnVlCiAgICAgIHJldHVybiAxMQogICAgZmkKICBkb25lCiAgcmV0dXJuIDEKfQoKcmVzdGFydHM9MApkZWxheT0yCgp3aGlsZSA6OyBkbwogIHQwPSRTRUNPTkRTCgogIHNydl9yYz0wCiAgYmFzaCAiJHtIRVJFfS9zZXJ2ZS5zaCIgIiRAIiA5PiYtIHx8IHNydl9yYz0kPwoKICBpZiAoKHNydl9yYyA9PSAwKSk7IHRoZW4KICAgIHJjPTAKICAgIHdhdGNoX3N0YWNrIHx8IHJjPSQ/CiAgICBjYXNlICIkcmMiIGluCiAgICAgIDEwfDExKSBsb2cgInN0b3BwZWQgY2xlYW5seSI7IGV4aXQgMCA7OwogICAgZXNhYwogICAgW1sgIiRyYyIgLWVxIDAgXV0gJiYgY29udGludWUKICBlbGlmICgoc3J2X3JjID09IDIpKTsgdGhlbgogICAgIyBVbnJlY292ZXJhYmxlIChlLmcuIHRoZSBwb3J0IGlzIGhlbGQgYnkgc29tZXRoaW5nIHRoYXQgaXMgbm90IG91cnMpLgogICAgIyBSZXN0YXJ0aW5nIHdvdWxkIGp1c3QgYnVybiB0aGUgcmV0cnkgYnVkZ2V0IGFuZCBoaWRlIHRoZSByZWFsIG1lc3NhZ2UuCiAgICBkaWUgImNhbm5vdCBzdGFydCBvbiBwb3J0ICR7UE9SVH07IHNlZSB0aGUgZXJyb3IgYWJvdmUiCiAgZWxzZQogICAgd2FybiAic2VydmUuc2ggZmFpbGVkIHRvIHN0YXJ0IChleGl0ICR7c3J2X3JjfSkiCiAgZmkKCiAgdXB0aW1lPSQoKFNFQ09ORFMgLSB0MCkpCiAgaWYgKCh1cHRpbWUgPj0gSEVBTFRIWV9BRlRFUikpOyB0aGVuCiAgICBkZWxheT0yCiAgICByZXN0YXJ0cz0wCiAgZWxzZQogICAgcmVzdGFydHM9JCgocmVzdGFydHMgKyAxKSkKICAgIGRlbGF5PSQoKGRlbGF5ICogMikpCiAgICAoKGRlbGF5ID4gTUFYX0JBQ0tPRkYpKSAmJiBkZWxheT0kTUFYX0JBQ0tPRkYKICBmaQoKICBpZiAoKHJlc3RhcnRzID4gTUFYX1JFU1RBUlRTKSk7IHRoZW4KICAgIGNzX2R1bXBfbG9nICIke1NEfS9jb2RlLXNlcnZlci5sb2ciIDQwCiAgICBkaWUgIiR7cmVzdGFydHN9IGZhaWxlZCByZXN0YXJ0cyBpbiBhIHJvdzsgZ2l2aW5nIHVwIgogIGZpCgogIHdhcm4gInJlc3RhcnRpbmcgaW4gJHtkZWxheX1zICh1cHRpbWUgJHt1cHRpbWV9cywgYXR0ZW1wdCAke3Jlc3RhcnRzfSkiCiAgc2xlZXAgIiRkZWxheSIKZG9uZQo='
    ),
    'stop.sh': (
        'IyEvdXNyL2Jpbi9lbnYgYmFzaAojCiMgY29sYWIvc3RvcC5zaCDigJQgc3RvcCBjb2RlLXNlcnZlciBhbmQgYW55IHR1bm5lbCBzdGFydGVkIGJ5IHNlcnZlLnNoLgojCiMgICBiYXNoIGNvbGFiL3N0b3Auc2gKIwpzZXQgLUVldW8gcGlwZWZhaWwKSEVSRT0iJChjZCAtLSAiJChkaXJuYW1lIC0tICIke0JBU0hfU09VUkNFWzBdfSIpIiAmJiBwd2QgLVApIgojIHNoZWxsY2hlY2sgc291cmNlPWxpYi9jb21tb24uc2gKc291cmNlICIke0hFUkV9L2xpYi9jb21tb24uc2giCgpTRD0iJChjc19zdGF0ZV9kaXIpIgpQT1JUPSIke0NTX1BPUlQ6LTgwODB9IgoKIyBUaGUgc3VwZXJ2aXNvciBtdXN0IGdvIGZpcnN0OiBpdCByZXN0YXJ0cyBzZXJ2ZS5zaCwgc28ga2lsbGluZyB0aGUgY2hpbGRyZW4KIyB3aGlsZSBpdCBzdGlsbCBydW5zIGp1c3QgZ2V0cyB0aGVtIHJlbGF1bmNoZWQgYSBmZXcgc2Vjb25kcyBsYXRlci4Kc3VwX3BpZD0iJChjc19yZWFkX3BpZCAiJHtTRH0vc3VwZXJ2aXNlLnBpZCIpIgppZiBbWyAtbiAiJHN1cF9waWQiIF1dICYmIGtpbGwgLTAgIiRzdXBfcGlkIiAyPi9kZXYvbnVsbDsgdGhlbgogIGxvZyAic3RvcHBpbmcgc3VwZXJ2aXNvciAocGlkICR7c3VwX3BpZH0pIgogICMgSXQgd2FzIGxhdW5jaGVkIHdpdGggc3RhcnRfbmV3X3Nlc3Npb24sIHNvIGl0cyBwaWQgaXMgYWxzbyBpdHMgcHJvY2VzcyBncm91cC4KICBraWxsIC1URVJNIC0tICItJHtzdXBfcGlkfSIgMj4vZGV2L251bGwgfHwga2lsbCAtVEVSTSAiJHN1cF9waWQiIDI+L2Rldi9udWxsIHx8IHRydWUKICBmb3IgXyBpbiB7MS4uMTV9OyBkbwogICAga2lsbCAtMCAiJHN1cF9waWQiIDI+L2Rldi9udWxsIHx8IGJyZWFrCiAgICBzbGVlcCAwLjMKICBkb25lCiAga2lsbCAtS0lMTCAtLSAiLSR7c3VwX3BpZH0iIDI+L2Rldi9udWxsIHx8IHRydWUKZWxpZiBbWyAiJHtDU19GUk9NX1NVUEVSVklTT1I6LTB9IiAhPSAiMSIgXV07IHRoZW4KICAjIE5vIHBpZGZpbGUgKG9sZGVyIHJ1biwgb3IgaXQgd2FzIGxvc3QpIOKAlCBtYXRjaCB0aGUgc3VwZXJ2aXNvcidzIG93biBhcmd2LgogIHBraWxsIC1mICdzdXBlcnZpc2VcLnNoJCcgMj4vZGV2L251bGwgfHwgdHJ1ZQpmaQpybSAtZiAtLSAiJHtTRH0vc3VwZXJ2aXNlLnBpZCIKCmNzX3N0b3BfcGlkZmlsZSAiJHtTRH0vY3MucGlkIiBjb2RlLXNlcnZlcgpjc19zdG9wX3BpZGZpbGUgIiR7U0R9L25ncm9rLnBpZCIgbmdyb2sKY3Nfc3RvcF9waWRmaWxlICIke1NEfS9jbG91ZGZsYXJlZC5waWQiIGNsb3VkZmxhcmVkCgojIEZhbGxiYWNrIGZvciBhIHJ1biB3aG9zZSBwaWRmaWxlIHdhcyBsb3N0LiBUaGUgcGF0dGVybnMgYXJlIGFuY2hvcmVkIHRvIHRoZQojIGZ1bGwgYXJndiBvbiBwdXJwb3NlOiB0aGlzIHJlcG9zaXRvcnkncyBkaXJlY3RvcnkgaXMgaXRzZWxmIG5hbWVkCiMgImNvZGUtc2VydmVyIiwgc28gYSBiYXJlIGBwa2lsbCAtZiBjb2RlLXNlcnZlcmAgd291bGQgYWxzbyBtYXRjaCDigJQgYW5kIGtpbGwg4oCUCiMgdGhlIHNoZWxsIHRoYXQgaW52b2tlZCB0aGlzIHNjcmlwdC4KIyBNYXRjaGVzIHRoZSByZWFsIGFyZ3Ygb2YgYSBjb2RlLXNlcnZlciBwcm9jZXNzICh0aGUgd3JhcHBlciByZS1leGVjcyB0bwojICJub2RlIC4uLi9vdXQvbm9kZS9lbnRyeS5qcyAtLWJpbmQtYWRkciAuLi4iKSwgbm90IGp1c3QgdGhlIHdyYXBwZXIgbmFtZS4KcGtpbGwgLWYgJ2VudHJ5XC5qcyAtLWJpbmQtYWRkcicgMj4vZGV2L251bGwgfHwgdHJ1ZQpwa2lsbCAtZiAnY29kZS1zZXJ2ZXIgLS1iaW5kLWFkZHInIDI+L2Rldi9udWxsIHx8IHRydWUKcGtpbGwgLWYgJ25ncm9rIGh0dHAnIDI+L2Rldi9udWxsIHx8IHRydWUKcGtpbGwgLWYgJ2Nsb3VkZmxhcmVkIHR1bm5lbCcgMj4vZGV2L251bGwgfHwgdHJ1ZQoKcm0gLWYgLS0gIiR7U0R9L2NzLnBpZCIgIiR7U0R9L25ncm9rLnBpZCIgIiR7U0R9L2Nsb3VkZmxhcmVkLnBpZCIKCiMgTGFzdCByZXNvcnQ6IHJlY2xhaW0gdGhlIHBvcnQgZnJvbSBhbiBvcnBoYW4gdGhhdCBubyBwaWRmaWxlIGtub3dzIGFib3V0LgppZiBjc19wb3J0X2luX3VzZSAiJFBPUlQiOyB0aGVuCiAgY3NfZnJlZV9wb3J0ICIkUE9SVCIgfHwgY3NfcG9ydF9yZXBvcnQgIiRQT1JUIgpmaQoKb2sgInN0b3BwZWQiCg=='
    ),
    'doctor.sh': (
        'IyEvdXNyL2Jpbi9lbnYgYmFzaAojCiMgY29sYWIvZG9jdG9yLnNoIOKAlCByZXBvcnQgdGhlIHN0YXRlIG9mIHRoZSBDb2xhYiBlbnZpcm9ubWVudCBhbmQgdGhlIHN0YWNrLgojCiMgICBiYXNoIGNvbGFiL2RvY3Rvci5zaAojCiMgUmVhZC1vbmx5OiBjaGFuZ2VzIG5vdGhpbmcuIFJ1biBpdCBiZWZvcmUgZmlsaW5nIGEgIml0IGRvZXNuJ3Qgd29yayIgbm90ZS4KIwpzZXQgLUVldW8gcGlwZWZhaWwKSEVSRT0iJChjZCAtLSAiJChkaXJuYW1lIC0tICIke0JBU0hfU09VUkNFWzBdfSIpIiAmJiBwd2QgLVApIgojIHNoZWxsY2hlY2sgc291cmNlPWxpYi9jb21tb24uc2gKc291cmNlICIke0hFUkV9L2xpYi9jb21tb24uc2giCgpTRD0iJChjc19zdGF0ZV9kaXIpIgpQT1JUPSIke0NTX1BPUlQ6LTgwODB9Igpwcm9ibGVtcz0wCgpiYWQoKSB7IHdhcm4gIiQqIjsgcHJvYmxlbXM9JCgocHJvYmxlbXMgKyAxKSk7IH0KCmt2KCkgeyBwcmludGYgJyAgICUtMjJzICVzXG4nICIkMSIgIiQyIjsgfQoKc2VjdGlvbigpIHsgcHJpbnRmICdcbiVzXG4nICIkKHByaW50ZiAnXDAzM1sxbSVzXDAzM1swbScgIiQqIikiOyB9CgpzZWN0aW9uICJob3N0IgprdiAiYXJjaGl0ZWN0dXJlIiAiJCh1bmFtZSAtbSkiCmt2ICJvcyIgIiQoY3Nfb3NfZmllbGQgUFJFVFRZX05BTUUpIgprdiAia2VybmVsIiAiJCh1bmFtZSAtcikiCmt2ICJ1c2VyIiAiJChpZCAtdW4pICh1aWQgJChpZCAtdSkpIgprdiAicHl0aG9uIiAiJChjb21tYW5kIC12IHB5dGhvbjMgPi9kZXYvbnVsbCAyPiYxICYmIHB5dGhvbjMgLVYgMj4mMSB8fCBlY2hvIG1pc3NpbmcpIgprdiAiYmFzaCIgIiRCQVNIX1ZFUlNJT04iCgpzZWN0aW9uICJyZXNvdXJjZXMiCm1lbV9nYj0iJChhd2sgJy9NZW1Ub3RhbC8geyBwcmludGYgIiUuMWYiLCAkMiAvIDEwNDg1NzYgfScgL3Byb2MvbWVtaW5mbyAyPi9kZXYvbnVsbCB8fCBlY2hvICc/JykiCmt2ICJtZW1vcnkiICIke21lbV9nYn0gR0IiCmlmIFtbIC1kIC9jb250ZW50IF1dOyB0aGVuCiAga3YgImRpc2sgL2NvbnRlbnQiICIkKGRmIC1oIC9jb250ZW50IDI+L2Rldi9udWxsIHwgYXdrICdOUj09MiB7IHByaW50ICQ0ICIgZnJlZSBvZiAiICQyIH0nKSIKZmkKa3YgInN0YXRlIGRpciIgIiR7U0R9IgprdiAic3RhdGUgc2l6ZSIgIiQoZHUgLXNoICIkU0QiIDI+L2Rldi9udWxsIHwgYXdrICd7IHByaW50ICQxIH0nIHx8IGVjaG8gJzAnKSIKCnNlY3Rpb24gImNvZGUtc2VydmVyIgppZiBjb21tYW5kIC12IGNvZGUtc2VydmVyID4vZGV2L251bGwgMj4mMTsgdGhlbgogIGt2ICJwYXRoIiAiJChjb21tYW5kIC12IGNvZGUtc2VydmVyKSIKICBrdiAidmVyc2lvbiIgIiQoY29kZS1zZXJ2ZXIgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgaGVhZCAtMSB8fCBlY2hvICc/JykiCiAgW1sgLWYgIiR7U0R9Ly5pbnN0YWxsZWQtdmVyc2lvbiIgXV0gJiYga3YgImluc3RhbGxlZCB0YWciICIkKGNhdCAiJHtTRH0vLmluc3RhbGxlZC12ZXJzaW9uIikiCmVsc2UKICBiYWQgImNvZGUtc2VydmVyIGlzIG5vdCBvbiBQQVRIIChydW46IGJhc2ggY29sYWIvaW5zdGFsbC5zaCkiCmZpCgpzZWN0aW9uICJ0dW5uZWxzIgppZiBjb21tYW5kIC12IG5ncm9rID4vZGV2L251bGwgMj4mMTsgdGhlbgogIGt2ICJuZ3JvayIgIiQobmdyb2sgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgaGVhZCAtMSkiCiAgaWYgbmdyb2sgY29uZmlnIGNoZWNrID4vZGV2L251bGwgMj4mMTsgdGhlbgogICAga3YgIm5ncm9rIGF1dGh0b2tlbiIgImNvbmZpZ3VyZWQiCiAgZWxzZQogICAgYmFkICJuZ3JvayBhdXRodG9rZW4gbm90IGNvbmZpZ3VyZWQgKG5ncm9rIGNvbmZpZyBhZGQtYXV0aHRva2VuIDx0b2tlbj4pIgogIGZpCmVsc2UKICBrdiAibmdyb2siICJub3QgaW5zdGFsbGVkIgpmaQppZiBjb21tYW5kIC12IGNsb3VkZmxhcmVkID4vZGV2L251bGwgMj4mMTsgdGhlbgogIGt2ICJjbG91ZGZsYXJlZCIgIiQoY2xvdWRmbGFyZWQgLS12ZXJzaW9uIDI+L2Rldi9udWxsIHwgYXdrICdOUj09MSB7IHByaW50ICQzIH0nKSIKZWxzZQogIGt2ICJjbG91ZGZsYXJlZCIgIm5vdCBpbnN0YWxsZWQiCmZpCgpzZWN0aW9uICJydW50aW1lIHN0YXRlIgppZiBjc19hbGl2ZSAiJHtTRH0vY3MucGlkIjsgdGhlbgogIGt2ICJjb2RlLXNlcnZlciIgInJ1bm5pbmcgKHBpZCAkKGNzX3JlYWRfcGlkICIke1NEfS9jcy5waWQiKSkiCmVsc2UKICBrdiAiY29kZS1zZXJ2ZXIiICJub3QgcnVubmluZyIKZmkKaWYgY3NfYWxpdmUgIiR7U0R9L25ncm9rLnBpZCI7IHRoZW4KICBrdiAibmdyb2siICJydW5uaW5nIChwaWQgJChjc19yZWFkX3BpZCAiJHtTRH0vbmdyb2sucGlkIikpIgpmaQppZiBjc19hbGl2ZSAiJHtTRH0vY2xvdWRmbGFyZWQucGlkIjsgdGhlbgogIGt2ICJjbG91ZGZsYXJlZCIgInJ1bm5pbmcgKHBpZCAkKGNzX3JlYWRfcGlkICIke1NEfS9jbG91ZGZsYXJlZC5waWQiKSkiCmZpCgprdiAicG9ydCAke1BPUlR9IiAiJChjc19wb3J0X2luX3VzZSAiJFBPUlQiICYmIGVjaG8gJ2luIHVzZScgfHwgZWNobyAnZnJlZScpIgpvd25lcnM9IiQoY3NfcG9ydF9vd25lcl9waWRzICIkUE9SVCIpIgpmb3IgcGlkIGluICRvd25lcnM7IGRvCiAga3YgInBvcnQgJHtQT1JUfSBvd25lciIgInBpZCAke3BpZH06ICQoY3NfY21kbGluZSAiJHBpZCIgfCBjdXQgLWMxLTcwKSIKZG9uZQprdiAiZXN0YWJsaXNoZWQgY29ubnMiICIkKGNzX2FjdGl2ZV9jb25ucyAiJFBPUlQiKSIKCmlmIGN1cmwgLWZzUyAtLW1heC10aW1lIDIgLS1ub3Byb3h5ICcqJyAiaHR0cDovLzEyNy4wLjAuMToke1BPUlR9L2hlYWx0aHoiID4vZGV2L251bGwgMj4mMTsgdGhlbgogIGt2ICJoZWFsdGh6IiAib2siCmVsc2UKICBrdiAiaGVhbHRoeiIgIm5vdCByZXNwb25kaW5nIgpmaQoKaWYgW1sgLXMgIiR7U0R9L2NyZWRlbnRpYWxzLnR4dCIgXV07IHRoZW4KICBzZWN0aW9uICJjcmVkZW50aWFscyAoZnJvbSB0aGUgbGFzdCBzZXJ2ZS5zaCBydW4pIgogICMgUGFzc3dvcmQgaW50ZW50aW9uYWxseSBub3QgcHJpbnRlZCBoZXJlOyByZWFkIHRoZSAwNjAwIGZpbGUgaWYgeW91IG5lZWQgaXQuCiAgc2VkIC1lICdzL15wYXNzd29yZD0uKi9wYXNzd29yZD08aGlkZGVuOyBzZWUgdGhlIGZpbGU+LycgIiR7U0R9L2NyZWRlbnRpYWxzLnR4dCIgXAogICAgfCBzZWQgJ3MvXi8gICAvJwpmaQoKc2VjdGlvbiAibG9ncyIKZm9yIGYgaW4gY29kZS1zZXJ2ZXIubG9nIG5ncm9rLmxvZyBjbG91ZGZsYXJlZC5sb2cgZXh0ZW5zaW9ucy5sb2c7IGRvCiAgaWYgW1sgLWYgIiR7U0R9LyR7Zn0iIF1dOyB0aGVuCiAgICBwcmludGYgJyAgICUtMjJzICVzIGJ5dGVzXG4nICIkZiIgIiQod2MgLWMgPCIke1NEfS8ke2Z9IiB8IHRyIC1kICcgJykiCiAgZmkKZG9uZQoKcHJpbnRmICdcbicKaWYgKChwcm9ibGVtcyA9PSAwKSk7IHRoZW4KICBvayAibm8gcHJvYmxlbXMgZGV0ZWN0ZWQiCmVsc2UKICB3YXJuICIke3Byb2JsZW1zfSBwcm9ibGVtKHMpIGRldGVjdGVkIgpmaQo='
    ),
    'README.md': (
        'IyBjb2RlLXNlcnZlciBvbiBHb29nbGUgQ29sYWIKClJ1bnMgY29kZS1zZXJ2ZXIgaW4gYSBDb2xhYiBWTSBhbmQgZXhwb3NlcyBpdCBvdmVyIGEgdHVubmVsLgoKKipObyBwYXRjaGVzIHRvIGNvZGUtc2VydmVyLioqIEl0IGlzIGEgcGxhaW4gTm9kZSBIVFRQIHNlcnZlcjsgbm90aGluZyBpbiBgc3JjL2AKb3IgYHBhdGNoZXMvYCBuZWVkcyB0byBjaGFuZ2UuIFRoZXNlIHNjcmlwdHMgb25seSBwaWNrIGZsYWdzIGFwcHJvcHJpYXRlIHRvIGFuCmVwaGVtZXJhbCBDb2xhYiBWTSwgaW5zdGFsbCB0aGUgcnVudGltZSBsaWJyYXJpZXMsIGFuZCBtYW5hZ2UgdGhlIHR1bm5lbC4KCmBgYApjb2xhYi8K4pSc4pSA4pSAIGluc3RhbGwuc2ggICAgICAgIyBsaWJyYXJpZXMgKyBjb2RlLXNlcnZlciB0YXJiYWxsIChjYWNoZWQpICsgdHVubmVsIGFnZW50CuKUnOKUgOKUgCBzZXJ2ZS5zaCAgICAgICAgICMgc3RhcnQgY29kZS1zZXJ2ZXIsIHdhaXQgZm9yIGhlYWx0aCwgb3BlbiB0dW5uZWwsIHJlcG9ydArilJzilIDilIAgc3VwZXJ2aXNlLnNoICAgICAjIHJlc3RhcnQgd2l0aCBiYWNrb2ZmICsgaWRsZSAvIG1heC1ydW50aW1lIHdhdGNoZG9nCuKUnOKUgOKUgCBzdG9wLnNoICAgICAgICAgICMgc3RvcCBldmVyeXRoaW5nCuKUnOKUgOKUgCBkb2N0b3Iuc2ggICAgICAgICMgcmVhZC1vbmx5IGRpYWdub3N0aWNzCuKUnOKUgOKUgCBsaWIvY29tbW9uLnNoICAgICMgc2hhcmVkIGhlbHBlcnMgKGxvZ2dpbmcsIGxvY2tzLCBwaWRmaWxlcywgcHJvYmVzKQrilJTilIDilIAgY29kZS1zZXJ2ZXIuaXB5bmIKYGBgCgojIyBXaGF0IGEgInRyYWNlIiBhY3R1YWxseSBpcywgYW5kIHdoYXQgYSB3cmFwcGVyIGNhbiBkbyBhYm91dCBpdAoKV29ydGggcmVhZGluZyBiZWZvcmUgYXNraW5nIGZvciBwcm9jZXNzIHJlbmFtaW5nLCBiZWNhdXNlIHRoZSBsYXllcnMgYXJlIG5vdAplcXVhbGx5IHJlYWNoYWJsZToKCnwgTGF5ZXIgfCBPYnNlcnZlZCBieSB8IENoYW5nZWFibGUgYnkgYSB3cmFwcGVyPyB8CnwtLS18LS0tfC0tLXwKfCBgL3Byb2MvPHBpZD4vY21kbGluZWAsIHByb2Nlc3MgbmFtZSB8IGFueXRoaW5nIGluc2lkZSB0aGUgVk0gfCB5ZXMsIHRyaXZpYWxseSDigJQgYW5kIGl0IGRvZXMgbm90IG1hdHRlcjogQ29sYWIncyBlbmZvcmNlbWVudCBpcyBub3QgYSBwZXItcHJvY2VzcyBzY2FubmVyLiBjb2RlLXNlcnZlcidzIG93biBjaGlsZHJlbiAoYG5vZGVgLCBleHRlbnNpb24gaG9zdHMsIGV2ZXJ5IHRlcm1pbmFsIHlvdSBvcGVuKSBrZWVwIHRoZSByZWFsIG5hbWVzLCBzbyBhIHJlbmFtZWQgcGFyZW50IGlzICppbmNvbnNpc3RlbnQqLCB3aGljaCBpcyBhIHdvcnNlIHNpZ25hbCB0aGFuIGFuIGhvbmVzdCBvbmUuIHwKfCBmaWxlc3lzdGVtIHdyaXRlcyB8IHRoZSBWTSBpdHNlbGYgfCB5ZXMsIHBvaW50bGVzcyDigJQgdGhlIGRpc2sgaXMgd2lwZWQgcGVyIHJ1bnRpbWUgfAp8IGVncmVzcyBTTkkgLyBob3N0bmFtZSAoYCoubmdyb2stZnJlZS5hcHBgKSB8IHRoZSBuZXR3b3JrIHBhdGggfCBvbmx5IGJ5IHVzaW5nIHlvdXIgb3duIGRvbWFpbiBvbiBhIG5hbWVkIHR1bm5lbC4gVGhhdCBpcyBub3QgZGlzZ3Vpc2UsIGl0IGlzIGhvc3Rpbmcg4oCUIHNlZSBiZWxvdy4gfAp8IHR1bm5lbC1zaWRlIEhUVFAvV1Mgc2hhcGUgKGAvd2Vic29ja2V0YCwgYXNzZXQgcGF0aHMpIHwgbmdyb2sgLyBDbG91ZGZsYXJlIGVkZ2UsIG5vdCBHb29nbGUgfCBubyByZWFzb24gdG8gY2FyZSB8CnwgcnVudGltZSBiZWhhdmlvdXI6IGlkbGUgR1BVIFZNLCBzZXNzaW9uIGxlbmd0aCwgemVybyBleGVjdXRlZCBjZWxscywgbWVtb3J5L0NQVSBwcm9maWxlIHwgR29vZ2xlLCBhY2NvdW50IGxldmVsIHwgKipubyoqIOKAlCBhbmQgdGhpcyBpcyB0aGUgbGF5ZXIgdGhhdCBnZXRzIGFjdGlvbmVkIHwKfCBhY2NvdW50IC8gc2Vzc2lvbiBjb3VudCAvIElQIHJlcHV0YXRpb24gfCBHb29nbGUgfCBubyB8CgpTbyB0aGUgdHdvIGxheWVycyBhIHdyYXBwZXIgKmNhbiogdG91Y2ggYXJlIHRoZSB0d28gbm9ib2R5IGVuZm9yY2VzIG9uLCBhbmQgdGhlCm9uZSB0aGF0IG1hdHRlcnMgaXMgdW50b3VjaGVkIGJ5IGFueSBzY3JpcHQuIFdoYXQgdGhpcyB0b29saW5nIGRvZXMgaW5zdGVhZCBpcwphZGRyZXNzIHRoZSB1bmRlcmx5aW5nIGNvbXBsYWludHMgcHJvcGVybHk6CgoqICoqcm90YXRpbmcgVVJMKiog4oaSIG5hbWVkIENsb3VkZmxhcmUgdHVubmVsIG9uIHlvdXIgb3duIGRvbWFpbiwgc3RhYmxlIGFjcm9zcwogIHJlc3RhcnRzIChgQ0ZfVE9LRU5gICsgYENGX1RVTk5FTF9VUkxgKSwgbm8gaW50ZXJzdGl0aWFsIHBhZ2UKKiAqKmJsYW5rIHBhZ2Ugb24gbmdyb2sqKiDihpIgcmVzZXJ2ZWQgbmdyb2sgZG9tYWluIChgTkdST0tfRE9NQUlOYCkKKiAqKlZNIGhlbGQgb3BlbiBmb3IgaG91cnMgZm9yIG5vdGhpbmcqKiDihpIgYHN1cGVydmlzZS5zaGAgaWRsZSBhbmQgbWF4LXJ1bnRpbWUKICB3YXRjaGRvZ3MuIFRoaXMgaXMgYWxzbyB0aGUgaG9uZXN0IGZpeCBmb3IgImxlc3Mgbm90aWNlYWJsZSI6IGEgR1BVIFZNIGlkbGluZwogIHdpdGggbm8gaW50ZXJhY3RpdmUgdHJhZmZpYyBpcyB0aGUgYWN0dWFsIHBhdHRlcm4gdGhhdCBnZXRzIHRocm90dGxlZCwgYW5kCiAgcmVsZWFzaW5nIGl0IGNvc3RzIG5vdGhpbmcuCgpUcmFmZmljIG9yIHByb2Nlc3MgcmVzcG9vZmluZyBpcyBkZWxpYmVyYXRlbHkgbm90IGltcGxlbWVudGVkLgoKIyMgUXVpY2sgc3RhcnQKCmBgYGJhc2gKZXhwb3J0IE5HUk9LX1RPS0VOPS4uLiAgICAgICAgIyBkYXNoYm9hcmQubmdyb2suY29tL2dldC1zdGFydGVkL3lvdXItYXV0aHRva2VuCmJhc2ggY29sYWIvaW5zdGFsbC5zaApiYXNoIGNvbGFiL3NlcnZlLnNoCmBgYAoKIyMgRmxvdyBhbmQgbG9ncwoKYGBgCmluc3RhbGwuc2ggICAgICAgICAgICAgICAgICAgc2xvdzogMeKAkzMgbWluIG9uIGEgZnJlc2ggcnVudGltZQogIGFwdCBydW50aW1lIGxpYnMg4oaSIGNvZGUtc2VydmVyIHRhcmJhbGwgKH4xMDAgTUIpIOKGkiBuZ3JvayDihpIgY2xvdWRmbGFyZWQKCnN1cGVydmlzZS5zaCAgICAgICAgICAgICAgICAgcmV0dXJucyBpbW1lZGlhdGVseSwgcnVucyBkZXRhY2hlZAogIOKUlOKUgCBzZXJ2ZS5zaAogICAgICAgY2xlYXIgc3RhbGUgcGlkZmlsZXMKICAgICAgIHBvcnQgY2hlY2sKICAgICAgIGNvZGUtc2VydmVyIOKGkiBwb2xsIC9oZWFsdGh6ICAgICAgICAgICAgNeKAkzkwIHMKICAgICAgIHR1bm5lbCAgICAgIOKGkiBwb2xsIGZvciBwdWJsaWMgVVJMICAgICAxMOKAkzYwIHMKICAgICAgIHdyaXRlIGNyZWRlbnRpYWxzLnR4dCDihpIgcHJpbnQgVVJMICsgcGFzc3dvcmQKICDilJTilIAgcmVzdGFydCBvbiBjcmFzaCwgc3RvcCBvbiBpZGxlIC8gbWF4IHJ1bnRpbWUKYGBgCgp8IEZpbGUgfCBDb250ZW50cyB8CnwtLS18LS0tfAp8IGAkQ1NfU1RBVEVfRElSL3N1cGVydmlzZS5vdXRgIHwgKipwcm9ncmVzcyBsb2cqKiwgb25lIGxpbmUgcGVyIHN0YWdlIHwKfCBgJENTX1NUQVRFX0RJUi9jb2RlLXNlcnZlci5sb2dgIHwgZWRpdG9yIHNlcnZlcjsgc3RhcnR1cCBjcmFzaGVzIGxhbmQgaGVyZSB8CnwgYCRDU19TVEFURV9ESVIvY2xvdWRmbGFyZWQubG9nYCB8IHR1bm5lbCByZWdpc3RyYXRpb24sIHB1YmxpYyBVUkwgfAp8IGAkQ1NfU1RBVEVfRElSL25ncm9rLmxvZ2AgfCBKU09OOyBsb29rIGZvciBgRVJSX05HUk9LXzQwMThgLCBgRVJSX05HUk9LXzEwOGAgfAp8IGAkQ1NfU1RBVEVfRElSL2NyZWRlbnRpYWxzLnR4dGAgfCBgdXJsPWAgLyBgcGFzc3dvcmQ9YCAvIGB0dW5uZWw9YCB8CgpIZWFsdGggYW5kIHR1bm5lbCB3YWl0cyBlbWl0IGEgaGVhcnRiZWF0IGV2ZXJ5IDEw4oCTMTUgcyAoYHdhaXRpbmcgZm9yIC4uLiAoMzBzKWApLApzbyBhIHNsb3cgc3RhcnQgaXMgZGlzdGluZ3Vpc2hhYmxlIGZyb20gYSBodW5nIG9uZS4gSWYgYHNlcnZlLnNoYCBzdG9wcyBtYWtpbmcKcHJvZ3Jlc3MsIHRoZSBsYXN0IGxpbmUgb2YgYHN1cGVydmlzZS5vdXRgIG5hbWVzIHRoZSBzdGFnZS4KCkZpcnN0IHJ1biBpcyBzbG93IGZvciByZWFzb25zIHRoYXQgYXJlIG5vdCBjb2RlLXNlcnZlcjogYGFwdC1nZXRgIG9uIGEgZnJlc2gKaW1hZ2UsIH4xNTAgTUIgb2YgZG93bmxvYWRzLCBhbmQgdHVubmVsIHJlZ2lzdHJhdGlvbi4gYGF1dG9gIHBpY2tzIG5ncm9rIHdoZW4gaXQKaGFzIGEgdG9rZW4gKGZhc3QpIGFuZCBjbG91ZGZsYXJlZCBvdGhlcndpc2UgKG5vIGFjY291bnQsIHNsb3dlciB0byByZWdpc3RlcikuCgojIyBUdW5uZWxzCgojIyMgbmdyb2sgKGRlZmF1bHQpCgpgYGBiYXNoCmJhc2ggY29sYWIvc2VydmUuc2ggICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgYXV0by1zZWxlY3RzIGEgdHVubmVsCkNTX1RVTk5FTD1uZ3JvayBiYXNoIGNvbGFiL3NlcnZlLnNoCk5HUk9LX0RPTUFJTj1teS1kZXYubmdyb2stZnJlZS5hcHAgYmFzaCBjb2xhYi9zZXJ2ZS5zaCAgICMgcmVzZXJ2ZWQsIHN0YWJsZQpOR1JPS19CQVNJQ19BVVRIPSd1c2VyOnBhc3MnIGJhc2ggY29sYWIvc2VydmUuc2ggICAgICAgICAjIGdhdGUgYXQgdGhlIGVkZ2UKYGBgCgpGcmVlIHRpZXIgc2hvd3MgYSAiWW91IGFyZSBhYm91dCB0byB2aXNpdCIgcGFnZSBvbiB0aGUgZmlyc3QgYnJvd3NlciBoaXQuIENsaWNrCnRocm91Z2ggaXQgYmVmb3JlIHRoZSBlZGl0b3IgbG9hZHMsIG90aGVyd2lzZSB0aGUgSlMgYXNzZXRzIGFyZSBzZXJ2ZWQgYXMgSFRNTAphbmQgeW91IGdldCBhIGJsYW5rIHBhZ2UuIEZyZWUgdGllciBhbHNvIGFsbG93cyBvbmUgYWdlbnQgc2Vzc2lvbiBhbmQgcm90YXRlcwp0aGUgaG9zdG5hbWUgdW5sZXNzIHlvdSByZXNlcnZlIGEgZG9tYWluLgoKIyMjIGNsb3VkZmxhcmVkIChubyBpbnRlcnN0aXRpYWwsIHN1cHBvcnRzIGEgcmVhbCBkb21haW4pCgpgYGBiYXNoCldJVEhfQ0xPVURGTEFSRUQ9MSBiYXNoIGNvbGFiL2luc3RhbGwuc2gKYmFzaCBjb2xhYi9zZXJ2ZS5zaCAtLXR1bm5lbD1jbG91ZGZsYXJlZCAgICAgICAgICAgICMgaHR0cHM6Ly88cmFuZD4udHJ5Y2xvdWRmbGFyZS5jb20KYGBgCgpOYW1lZCB0dW5uZWwsIHN0YWJsZSBob3N0bmFtZSwgbm8gd2FybmluZyBwYWdlOgoKYGBgYmFzaApjbG91ZGZsYXJlZCB0dW5uZWwgbG9naW4KY2xvdWRmbGFyZWQgdHVubmVsIGNyZWF0ZSBjb2xhYgpjbG91ZGZsYXJlZCB0dW5uZWwgcm91dGUgZG5zIGNvbGFiIGNvZGUuZXhhbXBsZS5jb20KIyB0aGVuLCBwZXIgc2Vzc2lvbjoKQ0ZfVE9LRU49PHR1bm5lbC10b2tlbj4gQ0ZfVFVOTkVMX1VSTD1odHRwczovL2NvZGUuZXhhbXBsZS5jb20gXAogIGJhc2ggY29sYWIvc2VydmUuc2ggLS10dW5uZWw9Y2xvdWRmbGFyZWQKYGBgCgpUaGlzIGlzIHRoZSBkdXJhYmxlIG9wdGlvbi4gTm90aGluZyBpcyBiZWluZyBoaWRkZW46IHRoZSB0cmFmZmljIGlzIGEgbm9ybWFsClRMUyBjb25uZWN0aW9uIHRvIGEgZG9tYWluIHlvdSBjb250cm9sLgoKIyMjIE5vdCBhbiBvcHRpb246IENvbGFiJ3Mgb3duIHBvcnQgcHJveHkKCmBnb29nbGUuY29sYWIub3V0cHV0LnNlcnZlX2tlcm5lbF9wb3J0X2FzX3dpbmRvdygpYCBjYW4gZXhwb3NlIGEgcG9ydCwgYnV0IHRoZQpwcm94eSBpcyBidWlsdCBhcm91bmQgSnVweXRlciBrZXJuZWwgY2hhbm5lbHMgYW5kIGRvZXMgbm90IHJlbGlhYmx5IGNhcnJ5IHRoZQpsb25nLWxpdmVkIGJpZGlyZWN0aW9uYWwgV2ViU29ja2V0IHRoZSBWUyBDb2RlIHdlYiBjbGllbnQgbmVlZHMuIFRoZSBlZGl0b3IgbWF5CmxvYWQgYW5kIHRoZW4gbG9zZSBpdHMgdGVybWluYWwuIFRoYXQgaXMgd2h5IGV2ZXJ5IHdvcmtpbmcgc2V0dXAgdXNlcyBhIHR1bm5lbC4KCiMjIFN1cGVydmlzaW9uCgpgYGBiYXNoCmJhc2ggY29sYWIvc3VwZXJ2aXNlLnNoCmBgYAoKKiByZXN0YXJ0cyBgc2VydmUuc2hgIHdpdGggZXhwb25lbnRpYWwgYmFja29mZiAoMiBzIOKGkiA2MCBzKSBhZnRlciBhIGNyYXNoCiogcmVzZXRzIHRoZSBiYWNrb2ZmIG9uY2UgdGhlIHN0YWNrIGhhcyBiZWVuIHVwIGZvciBgQ1NfSEVBTFRIWV9BRlRFUmAgc2Vjb25kcwoqIHNodXRzIGRvd24gYWZ0ZXIgYENTX0lETEVfVElNRU9VVGAgc2Vjb25kcyB3aXRoIHplcm8gZXN0YWJsaXNoZWQgY29ubmVjdGlvbnMKKiBzaHV0cyBkb3duIGFmdGVyIGBDU19NQVhfUlVOVElNRWAgc2Vjb25kcywgYWhlYWQgb2YgQ29sYWIncyBvd24gaGFyZCBjYXAKCiMjIENvbmZpZ3VyYXRpb24KCnwgVmFyaWFibGUgfCBEZWZhdWx0IHwgTWVhbmluZyB8CnwtLS18LS0tfC0tLXwKfCBgTkdST0tfVE9LRU5gIHwg4oCUIHwgbmdyb2sgYXV0aHRva2VuLCBvciBhZGQgaXQgb25jZSB3aXRoIGBuZ3JvayBjb25maWcgYWRkLWF1dGh0b2tlbmAgfAp8IGBOR1JPS19ET01BSU5gIHwg4oCUIHwgcmVzZXJ2ZWQgZG9tYWluIOKGkiBzdGFibGUgaG9zdG5hbWUsIG5vIHJvdGF0aW9uIHwKfCBgTkdST0tfQkFTSUNfQVVUSGAgfCDigJQgfCBgdXNlcjpwYXNzYCwgcmVqZWN0cyBzY2FubmVycyBhdCBuZ3JvaydzIGVkZ2UgfAp8IGBDRl9UT0tFTmAgfCDigJQgfCBDbG91ZGZsYXJlIG5hbWVkLXR1bm5lbCB0b2tlbiB8CnwgYENGX1RVTk5FTF9VUkxgIHwg4oCUIHwgcHVibGljIGhvc3RuYW1lIHRvIHJlcG9ydCBmb3IgYSBuYW1lZCB0dW5uZWwgfAp8IGBDU19UVU5ORUxgIHwgYGF1dG9gIHwgYGF1dG9gIFx8IGBuZ3Jva2AgXHwgYGNsb3VkZmxhcmVkYCBcfCBgbm9uZWAuIGBhdXRvYCA9IG5ncm9rIHdoZW4gaXQgaGFzIGNyZWRlbnRpYWxzLCBlbHNlIGEgY2xvdWRmbGFyZWQgcXVpY2sgdHVubmVsIChubyBhY2NvdW50IG5lZWRlZCkgfAp8IGBDU19QT1JUYCB8IGA4MDgwYCB8IGxvY2FsIHBvcnQgKG5ldmVyIHB1YmxpYzsgdGhlIGFnZW50IHJ1bnMgaW4gdGhlIHNhbWUgVk0pIHwKfCBgQ1NfUEFTU1dPUkRgIHwgZ2VuZXJhdGVkIHwgZml4ZWQgcGFzc3dvcmQ7IG90aGVyd2lzZSBwZXJzaXN0ZWQgdG8gdGhlIHN0YXRlIGRpciB8CnwgYENTX1NUQVRFX0RJUmAgfCBgL2NvbnRlbnQvLmNvZGUtc2VydmVyYCB8IGRhdGEsIGV4dGVuc2lvbnMsIGxvZ3MsIHBpZGZpbGVzLCBwYXNzd29yZCB8CnwgYENTX0NBQ0hFX0RJUmAgfCBgJENTX1NUQVRFX0RJUi9jYWNoZWAgfCByZWxlYXNlIHRhcmJhbGwgY2FjaGUg4oCUIHBvaW50IGF0IERyaXZlIHRvIHNraXAgcmUtZG93bmxvYWRzIHwKfCBgQ1NfVkVSU0lPTmAgfCBsYXRlc3QgfCBwaW4gYSBjb2RlLXNlcnZlciB2ZXJzaW9uIHwKfCBgQ1NfUFJFRklYYCB8IGAvdXNyL2xvY2FsYCB8IGluc3RhbGwgcHJlZml4IHwKfCBgQ1NfRVhURU5TSU9OU2AgfCDigJQgfCBjb21tYS1zZXBhcmF0ZWQgZXh0ZW5zaW9uIElEcywgaW5zdGFsbGVkIG9uY2UgfAp8IGBDU19JRExFX1RJTUVPVVRgIHwgYDE4MDBgIHwgc3VwZXJ2aXNlOiBpZGxlIHNlY29uZHMgYmVmb3JlIHNodXRkb3duIHwKfCBgQ1NfTUFYX1JVTlRJTUVgIHwgYDM5NjAwYCB8IHN1cGVydmlzZTogdG90YWwgc2Vjb25kcyBiZWZvcmUgc2h1dGRvd24gfAp8IGBDU19NQVhfUkVTVEFSVFNgIHwgYDIwYCB8IHN1cGVydmlzZTogY29uc2VjdXRpdmUgZmFpbHVyZXMgYmVmb3JlIGdpdmluZyB1cCB8CnwgYENTX0FMTE9XX0lOU0VDVVJFYCB8IGAwYCB8IG11c3QgYmUgYDFgIHRvIGFjY2VwdCBgLS1uby1hdXRoYCB8CnwgYENTX1NUQVRFX0RJUmAgb24gRHJpdmUgfCDigJQgfCBgQ1NfU1RBVEVfRElSPS9jb250ZW50L2RyaXZlL015RHJpdmUvLmNzYCBrZWVwcyBleHRlbnNpb25zIGFjcm9zcyBydW50aW1lcyBhdCBzb21lIEZTIGxhdGVuY3kgY29zdCB8CgojIyBEaWFnbm9zdGljcwoKYGBgYmFzaApiYXNoIGNvbGFiL2RvY3Rvci5zaApgYGAKClJlcG9ydHMgYXJjaGl0ZWN0dXJlLCBPUywgbWVtb3J5LCBgL2NvbnRlbnRgIGZyZWUgc3BhY2UsIGNvZGUtc2VydmVyIHZlcnNpb24sCnR1bm5lbCBjcmVkZW50aWFscywgbGl2ZSBwcm9jZXNzZXMsIHBvcnQgc3RhdGUsIGVzdGFibGlzaGVkIGNvbm5lY3Rpb25zLApgL2hlYWx0aHpgLCBsb2cgc2l6ZXMuIFJlYWQtb25seS4gVGhlIHBhc3N3b3JkIGlzIGhpZGRlbiBpbiBpdHMgb3V0cHV0OyByZWFkCmAkQ1NfU1RBVEVfRElSL2NyZWRlbnRpYWxzLnR4dGAgKG1vZGUgMDYwMCkgaWYgeW91IG5lZWQgaXQuCgojIyBXaHkgdGhlc2UgZmxhZ3MKCnwgRmxhZyB8IFJlYXNvbiB8CnwtLS18LS0tfAp8IGAtLWJpbmQtYWRkciAxMjcuMC4wLjE6UE9SVGAgfCB0aGUgdHVubmVsIGFnZW50IGlzIGluIHRoZSBzYW1lIFZNOyBub3RoaW5nIG5lZWRzIGEgcHVibGljIGJpbmQgfAp8IGAtLWF1dGggcGFzc3dvcmRgIHwgdHVubmVsIGhvc3RuYW1lcyBnZXQgcHJvYmVkIHdpdGhpbiBtaW51dGVzIOKAlCBuZXZlciBgLS1hdXRoIG5vbmVgIHwKfCBgLS1kaXNhYmxlLXRlbGVtZXRyeWAgLyBgLS1kaXNhYmxlLXVwZGF0ZS1jaGVja2AgfCBub2lzZSwgcGx1cyB3cml0ZXMgdG8gYW4gZXBoZW1lcmFsIGRpc2sgfAp8IGAtLWRpc2FibGUtd29ya3NwYWNlLXRydXN0YCB8IG90aGVyd2lzZSB0aGUgdHJ1c3QgbW9kYWwgZWF0cyB0aGUgZmlyc3QgY2xpY2sgfAp8IGAtLXVzZXItZGF0YS1kaXJgIC8gYC0tZXh0ZW5zaW9ucy1kaXJgIHwga2VlcHMgc3RhdGUgaW4gb25lIHBlcnNpc3RhYmxlIGRpcmVjdG9yeSB8CnwgYC0tdHJ1c3RlZC1vcmlnaW5zICcqJ2AgfCBvbmx5IGlmIGEgcHJveHkgKipyZXdyaXRlcyoqIGBIb3N0YCAoYC0taG9zdC1oZWFkZXI9cmV3cml0ZWAsIENhZGR5IGBoZWFkZXJfdXAgSG9zdGApLiBUaGF0IGlzIHRoZSBvbmUgY2FzZSB0aGF0IGxvb2tzIGxpa2UgaXQgbmVlZHMgYSBwYXRjaCBhbmQgZG9lcyBub3QuIHwKCiMjIFRyb3VibGVzaG9vdGluZwoKfCBTeW1wdG9tIHwgQ2F1c2UgfAp8LS0tfC0tLXwKfCBibGFuayBwYWdlLCBubyBlZGl0b3IgfCBuZ3JvayB3YXJuaW5nIHBhZ2Ugbm90IGNsaWNrZWQgdGhyb3VnaCB8CnwgYEVSUl9OR1JPS180MDE4YCB8IG5vIGF1dGh0b2tlbjogYG5ncm9rIGNvbmZpZyBhZGQtYXV0aHRva2VuIDx0b2tlbj5gIHwKfCBgRVJSX05HUk9LXzEwOGAgfCBmcmVlIHRpZXIgYWxsb3dzIG9uZSBhZ2VudCBzZXNzaW9uOyBgYmFzaCBjb2xhYi9zdG9wLnNoYCBmaXJzdCB8Cnwgc3RhbGUgNDA0IFVSTCB8IHByZXZpb3VzIHJ1bnRpbWUncyBVUkw7IHJlLXJ1biBgc2VydmUuc2hgIHwKfCBlZGl0b3IgbG9hZHMsIHRlcm1pbmFsIGRpZXMgfCBXZWJTb2NrZXQgbm90IGNhcnJpZWQgYnkgdGhlIHBhdGggaW4gdXNlIHwKfCBgcG9ydCA4MDgwIGlzIGJ1c3lgIHwgYW4gb3JwaGFuIGZyb20gYW4gZWFybGllciBydW47IGBzZXJ2ZS5zaGAgcmVjbGFpbXMgaXQgYXV0b21hdGljYWxseSB3aGVuIGl0IGxvb2tzIGxpa2Ugb3Vycy4gSWYgaXQgY2Fubm90LCB0aGUgcGlkIGFuZCBhcmd2IGFyZSBwcmludGVkIOKAlCBmcmVlIGl0IG9yIHVzZSBgLS1wb3J0PTxuPmAgfAp8IHJlc3RhcnQgbG9vcCwgbmV2ZXIgcmVhY2hpbmcgYSBVUkwgfCBhIHByZXZpb3VzIHN1cGVydmlzb3IgaXMgc3RpbGwgYWxpdmUgcmVsYXVuY2hpbmcgdGhlIHN0YWNrLiBgc3RvcC5zaGAga2lsbHMgdGhlIHN1cGVydmlzb3IgKmZpcnN0KiwgdGhlbiByZWNsYWltcyB0aGUgcG9ydCB8Cnwgc3R1Y2sgb24gYHdhaXRpbmcgZm9yIGNsb3VkZmxhcmVkIFVSTGAgfCBxdWljay10dW5uZWwgcmVnaXN0cmF0aW9uIGlzIHNsb3cgb3IgcmF0ZS1saW1pdGVkOyBzZXQgYE5HUk9LX1RPS0VOYCB8Cnwgc3R1Y2sgb24gYHdhaXRpbmcgZm9yIC4uLi9oZWFsdGh6YCB8IHN0YXJ0dXAgY3Jhc2gg4oCUIHJlYWQgYGNvZGUtc2VydmVyLmxvZ2AgfAp8IGBleHRlbnNpb25zIGFscmVhZHkgaW5zdGFsbGVkYCBidXQgbWlzc2luZyB8IGRlbGV0ZSBgJENTX1NUQVRFX0RJUi8uZXh0ZW5zaW9ucy0qYCBhbmQgcmVzdGFydCB8CgojIyBIYXJkZW5pbmcgbm90ZXMKCiogVGhlIHBhc3N3b3JkIGZpbGUgaXMgYDA2MDBgOyBgY3JlZGVudGlhbHMudHh0YCBpcyBgMDYwMGAgYW5kIGhvbGRzIHRoZQogIHBsYWludGV4dC4gRm9yIGEgc3Ryb25nZXIgc2V0dXAgdXNlIGNvZGUtc2VydmVyJ3MgYEhBU0hFRF9QQVNTV09SRGAKICAoYXJnb24yaWQpIGluIGB+Ly5jb25maWcvY29kZS1zZXJ2ZXIvY29uZmlnLnlhbWxgIGFuZCBzZXQgYENTX1BBU1NXT1JEYCB0bwogIG5vdGhpbmcsIGFjY2VwdGluZyB0aGF0IHRoZSBwYXNzd29yZCBpcyB0aGVuIG5vdCByZWNvdmVyYWJsZSBmcm9tIGRpc2suCiogYE5HUk9LX0JBU0lDX0FVVEhgIHB1dHMgYSBzZWNvbmQgY3JlZGVudGlhbCBpbiBmcm9udCBvZiBjb2RlLXNlcnZlciBzbyBwb3J0CiAgc2Nhbm5lcnMgbmV2ZXIgcmVhY2ggaXQuIENoZWFwIGFuZCB3b3J0aCBpdCBvbiBhIGZyZWUgaG9zdG5hbWUuCiogTmV2ZXIgY29tYmluZSBhIHB1YmxpYyB0dW5uZWwgd2l0aCBgLS1uby1hdXRoYCwgYW5kIG5ldmVyIGJpbmQgYDAuMC4wLjBgIG9uIGEKICBob3N0IHdpdGggYSBwdWJsaWMgaW50ZXJmYWNlLgoKIyMgT24gdGhlIENvbGFiIHRlcm1zCgpDb2xhYidzIEZBUSBkb2VzIG5vdCBsaXN0IHJlbW90ZS1kZXNrdG9wIG9yIHJlbW90ZS1JREUgdXNlIGFzIHN1cHBvcnRlZC4gVGhpcwp3b3JrcywgYnV0IHRoZSBydW50aW1lIGlzIGVwaGVtZXJhbCwgdGhlIFVSTCBjaGFuZ2VzIG9uIHJlc3RhcnQgdW5sZXNzIHlvdSB1c2UgYQpuYW1lZCB0dW5uZWwsIGFuZCBhY2NvdW50LWxldmVsIHRocm90dGxpbmcgaXMgYWx3YXlzIHBvc3NpYmxlIHJlZ2FyZGxlc3Mgb2YgaG93CnRoZSBwcm9jZXNzIGlzIG5hbWVkIG9yIHdyYXBwZWQuCg=='
    ),
}

(BASE / "lib").mkdir(parents=True, exist_ok=True)
for rel, blob in FILES.items():
    f = BASE / rel
    f.write_bytes(base64.b64decode(blob))
    if f.suffix == ".sh":
        f.chmod(f.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

print(f"wrote {len(FILES)} files to {BASE}")
for rel in sorted(FILES):
    print("  ", rel, (BASE / rel).stat().st_size, "bytes")

## 4. Install

Slow on purpose — this is where the 1–3 minutes go on a fresh runtime:
`apt-get` for the VS Code web client libraries, then the code-server tarball
(~100 MB), then both tunnel agents. Output streams live.

Re-running it is cheap: the tarball is cached in `CS_CACHE_DIR`, and apt is
skipped when the version already matches.

In [ ]:
import os, subprocess

env = dict(os.environ, WITH_CLOUDFLARED="1")  # cloudflared is the no-account fallback
subprocess.run(["bash", str(BASE / "install.sh")], check=True, env=env)

## 5. Start

Returns in about two seconds — the supervisor runs detached, so this cell does
not block and you are free to run the next one immediately.

In [ ]:
import os, signal, time, pathlib, subprocess

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
STATE.mkdir(parents=True, exist_ok=True)

# Re-running must restart cleanly: retire the previous supervisor (it holds a
# lock), then let stop.sh release the port.
pid_file = STATE / "supervise.pid"
if pid_file.exists():
    try:
        pid = int(pid_file.read_text().strip())
        os.killpg(os.getpgid(pid), signal.SIGTERM)
        print("stopped previous supervisor", pid)
        for _ in range(20):
            time.sleep(0.5)
            os.kill(pid, 0)
    except (ValueError, ProcessLookupError, PermissionError):
        pass
    pid_file.unlink(missing_ok=True)
subprocess.run(["bash", str(BASE / "stop.sh")], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

creds = STATE / "credentials.txt"
creds.unlink(missing_ok=True)          # only accept a fresh write

log_path = STATE / "supervise.out"
fh = open(log_path, "w")
proc = subprocess.Popen(
    ["bash", str(BASE / "supervise.sh")],
    stdout=fh, stderr=subprocess.STDOUT,
    start_new_session=True,            # own process group, so stop can kill it cleanly
)
pid_file.write_text(str(proc.pid))

print(f"supervisor started (pid {proc.pid}), detached")
print(f"progress log : {log_path}")
print("next         : run the cell below for live progress and the URL")

## 6. Wait for the URL (live progress)

Streams the tail of `supervise.out` while the tunnel comes up, so a slow start is
visible instead of looking hung. Typical first run: health ~10 s, then 10–60 s
for the tunnel depending on provider.

In [ ]:
import os, time, pathlib
from IPython.display import display, clear_output, HTML

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
creds = STATE / "credentials.txt"
log_path = STATE / "supervise.out"
pid_file = STATE / "supervise.pid"

BUDGET = 300
url = pw = None
reason = "timed out"

for elapsed in range(BUDGET):
    info = {}
    if creds.exists():
        raw = creds.read_text().strip()
        if "url=" in raw:
            info = dict(l.split("=", 1) for l in raw.splitlines() if "=" in l)
            if info.get("url"):
                url, pw, reason = info["url"], info.get("password"), "ok"
                break

    tail = ""
    if log_path.exists():
        tail = "\n".join(log_path.read_text().splitlines()[-10:])

    alive = True
    if pid_file.exists():
        try:
            os.kill(int(pid_file.read_text().strip()), 0)
        except (ValueError, ProcessLookupError, PermissionError):
            alive = False

    clear_output(wait=True)
    if not alive and not url:
        reason = "supervisor exited"
        break
    print(f"waiting {elapsed}s …  (log: {log_path})")
    print("─" * 60)
    print(tail or "(no output yet)")

    time.sleep(1)

clear_output(wait=True)
if url:
    display(HTML(
        f'<p style="font-size:17px;margin:8px 0"><b>code-server</b> &nbsp;'
        f'<a href="{url}" target="_blank">{url}</a></p>'
        f'<p style="margin:4px 0"><b>password</b> &nbsp;'
        f'<code style="font-size:15px">{pw}</code></p>'
    ))
    print(f"tunnel : {info.get('tunnel')}")
    print(f"state  : {STATE}")
    if info.get("tunnel") == "ngrok":
        print()
        print("ngrok free tier: click through the warning page before the editor loads.")
else:
    print(f"no URL yet ({reason}). Full progress log:\n")
    print(log_path.read_text() if log_path.exists() else "(no log)")
    print("\nNext: cell 8 (Status) for diagnostics, or cell 9 (Stop) then cell 6 again.")

## 7. Status

In [ ]:
import pathlib
STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
supervise_log = STATE / "supervise.out"

if supervise_log.exists():
    print("── supervisor ─────────────────────────────────────────────")
    print("\n".join(supervise_log.read_text().splitlines()[-12:]))
    print()

!bash "{BASE}/doctor.sh"

## 8. Stop

In [ ]:
import os, pathlib, signal

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
pid_file = STATE / "supervise.pid"
if pid_file.exists():
    try:
        pid = int(pid_file.read_text().strip())
        os.killpg(os.getpgid(pid), signal.SIGTERM)   # supervisor was started detached
        print("stopped supervisor", pid)
    except (ValueError, ProcessLookupError, PermissionError):
        pass
    pid_file.unlink(missing_ok=True)

!bash "{BASE}/stop.sh"

## Troubleshooting

| Symptom | Cause / fix |
|---|---|
| cell 7 sits on "waiting for cloudflared URL" | Cloudflare quick-tunnel registration is slow or rate-limited; set `NGROK_TOKEN` for the faster path |
| stuck at `waiting for http://127.0.0.1:8080/healthz` | startup crash — read `code-server.log` |
| `port 8080 in use` | orphaned process: cell 9, then cell 6 |
| blank page after opening the URL | ngrok interstitial not clicked through, or set `TUNNEL = "cloudflared"` |
| URL 404s / changes | free ngrok rotates hostnames; set `NGROK_DOMAIN`, or use a named cloudflared tunnel |
| extensions missing | `rm -rf /content/.code-server/.extensions-*`, restart |

Useful from a shell cell:

```bash
tail -n 60 /content/.code-server/supervise.out
tail -n 60 /content/.code-server/code-server.log
bash /content/colab-cs/doctor.sh
```

The readable sources live in the repo at `colab/`; this notebook is generated
from them by `colab/build-notebook.py`, so prefer editing there.